# Can the equations of a physical reservoir predict its computing capacity?

> **Work in progress, not peer reviewed.** See the README.

Run the cells in order: each cell uses functions defined by the cells before it. Each cell prints its debugging milestones as `[OK]` / `[FAIL]` lines (comments and messages are in French).

Generated from `src/` by `tools/build_notebook.py`.

## Step 1 — `complexity_vector.py`

complexity vector (8 axes)

In [ ]:
"""
Étape 1 — Le vecteur de complexité C d'une dynamique.

Entrée : une série d'états X de forme (T, N) : T pas de temps, N canaux lus.
Sortie : un dictionnaire avec les composantes de C.

    H_perm   entropie de permutation normalisée (Bandt-Pompe)      [0, 1]
    C_JS     complexité statistique de Jensen-Shannon (Rosso)       [0, ~0.5]
    PR       dimension effective (rapport de participation) / N     [1/N, 1]
    C_N      complexité neurale gaussienne (Tononi-Sporns-Edelman)  >= 0
    C_N_form C_N rapporté à son maximum possible (forme de l'intégration) [0, 1]
    tau_med  temps de relaxation médian (en pas)                    > 0
    tau_sprd dispersion des temps de relaxation (écart-type de log tau)
    lam      exposant de Lyapunov conditionnel (si 2e trajectoire)  <0 : écho, >0 : chaos
    consist  consistance (si 3e trajectoire partie d'ailleurs)      1 : oublie son départ

Lancer `python complexity_vector.py` exécute les tests de validation (milestones).
"""

import time
from math import factorial

import numpy as np


# ---------------------------------------------------------------- milestones
_T0 = time.perf_counter()


def milestone(tag, ok, msg):
    """Affiche une étape de debug : [M1] [OK]  message   (temps écoulé)."""
    status = "OK  " if ok else "FAIL"
    print(f"[{tag}] [{status}] {msg:<62} ({time.perf_counter() - _T0:5.2f} s)")
    return ok


# ------------------------------------------------- 1. Bandt-Pompe : H et C_JS
def permutation_probs(x, D=5, tau=1, warn=True, rng=None):
    """Probabilités des motifs ordinaux de longueur D dans la série x.

    Égalités (données quantifiées, ex. 6-10 bits) : elles sont départagées au
    hasard. Sinon argsort les classe toujours dans le même ordre, ce qui crée
    de faux motifs favoris et gonfle C_JS (bruit sur 4 bits : C x48).
    """
    n = len(x) - (D - 1) * tau
    if warn and n < 10 * factorial(D):
        print(f"   ! attention : {n} fenêtres pour {factorial(D)} motifs, statistique faible")
    ties = np.mean(np.diff(x) == 0)
    if ties > 0:
        if warn and ties > 0.01:
            print(f"   ! attention : {ties:.1%} de valeurs consécutives égales, départagées au hasard")
        rng = np.random.default_rng(0) if rng is None else rng
        steps = np.diff(np.unique(x))
        x = x + rng.uniform(-0.25, 0.25, len(x)) * (steps.min() if len(steps) else 1.0)
    windows = np.stack([x[i * tau: i * tau + n] for i in range(D)], axis=1)
    patterns = np.argsort(windows, axis=1, kind="stable")
    codes = patterns @ (D ** np.arange(D))          # un entier par motif
    _, counts = np.unique(codes, return_counts=True)
    p = np.zeros(factorial(D))
    p[: len(counts)] = counts / counts.sum()        # motifs absents = 0
    return p


def _shannon(p):
    p = p[p > 0]
    return -np.sum(p * np.log(p))


def entropy_complexity(x, D=5, tau=1, warn=True):
    """Retourne (H normalisée, C_JS) pour une série 1D (Rosso et al. 2007)."""
    p = permutation_probs(x, D, tau, warn)
    M = len(p)
    pe = np.full(M, 1.0 / M)
    H = _shannon(p) / np.log(M)
    J = _shannon((p + pe) / 2) - _shannon(p) / 2 - _shannon(pe) / 2
    Q0 = -2.0 / (((M + 1) / M) * np.log(M + 1) - 2 * np.log(2 * M) + np.log(M))
    return H, Q0 * J * H


# ------------------------------------------------- 2. dimension effective
def participation_ratio(X, standardize=True):
    """(Σλ)² / Σλ² des valeurs propres, divisé par N.

    standardize=True : matrice de corrélation, insensible aux unités des canaux
    (sinon un canal en mV et un autre en V ne pèsent pas pareil).
    standardize=False : covariance brute, utile si tous les canaux ont la même
    unité et qu'un canal presque muet ne doit pas compter comme une direction.
    """
    M = np.corrcoef(X, rowvar=False) if standardize else np.cov(X, rowvar=False)
    lam = np.clip(np.linalg.eigvalsh(M), 0, None)
    return (lam.sum() ** 2 / (lam ** 2).sum()) / X.shape[1]


# ------------------------------------------------- 3. temps de relaxation
def timescales(X, max_lag=500, warn=True):
    """Pour chaque canal : retard après lequel |autocorrélation| ne remonte plus
    au-dessus de 1/e.

    On prend le DERNIER passage et la valeur absolue : un canal qui oscille
    (ex. mode de giration d'un skyrmion) passe vite sous 1/e mais y remonte à
    chaque période. Le premier passage le prendrait à tort pour un canal sans
    mémoire. Pour une décroissance monotone, les deux définitions coïncident.
    """
    taus, capped = [], 0
    for x in X.T:
        x = x - x.mean()
        f = np.fft.rfft(x, n=2 * len(x))
        acf = np.fft.irfft(f * np.conj(f))[: max_lag + 1]
        acf = acf / acf[0]
        above = np.where(np.abs(acf) >= 1 / np.e)[0]
        tau = above[-1] + 1
        capped += tau > max_lag
        taus.append(min(tau, max_lag))
    if warn and capped:
        print(f"   ! attention : {capped} canal(aux) gardent leur mémoire au-delà de "
              f"max_lag={max_lag}, tau sous-estimé")
    taus = np.array(taus, float)
    return np.median(taus), np.std(np.log(taus))


# ------------------------------------------------- 4. complexité neurale
def _integration(R):
    """I(X) = -1/2 log det R pour des variables gaussiennes."""
    sign, logdet = np.linalg.slogdet(R)
    return -0.5 * logdet


def neural_complexity(X, n_max=24, n_samples=30, noise_floor=0.01, rng=None):
    """C_N = Σ_k [ (k/n) I(X) - <I(X_k)> ], sous-ensembles tirés au hasard.

    noise_floor : bruit de mesure, en fraction de l'écart-type de chaque canal
    (0.01 = 1 %, environ 7 bits). La matrice utilisée est exactement celle des
    canaux + ce bruit : les directions sous le plancher ne comptent plus.

    Renvoie (C_N brut, forme). La forme = C_N / ((n-1)/2 · I(X)) est dans [0, 1] :
    C_N brut grandit aussi avec la synchronie (49 pour 20 canaux synchronisés,
    contre 4.7 pour X_mix), la forme non (0.06 contre 0.16). C'est la forme qui
    est « basse aux deux extrêmes ».
    """
    rng = np.random.default_rng(0) if rng is None else rng
    n = min(X.shape[1], n_max)
    cols = rng.choice(X.shape[1], n, replace=False)
    v = noise_floor ** 2
    R = (np.corrcoef(X[:, cols], rowvar=False) + v * np.eye(n)) / (1 + v)
    I_total = _integration(R)
    cn = 0.0
    for k in range(1, n):                           # k = n donne 0 : inutile
        Ik = np.mean([_integration(R[np.ix_(s, s)])
                      for s in (rng.choice(n, k, replace=False) for _ in range(n_samples))])
        cn += (k / n) * I_total - Ik
    # Avec T pas, des canaux indépendants montrent quand même une intégration
    # accidentelle d'environ n(n-1)/(4T). En dessous de 5 fois ce biais, il n'y a
    # rien à mettre en forme : le rapport de deux quasi-zéros serait du bruit.
    bias = n * (n - 1) / (4 * X.shape[0])
    form = cn / ((n - 1) / 2 * I_total) if I_total > 5 * bias else 0.0
    return cn, float(np.clip(form, 0, 1))


# ------------------------------------------------- 5. Lyapunov conditionnel
def conditional_lyapunov(Xa, Xb, floor=1e-10, ceil_frac=0.1):
    """Pente de log||Xa - Xb|| : deux trajectoires, même entrée, départs différents.

    Unité : par pas d'échantillonnage. Fenêtre d'ajustement : entre `floor`
    (erreurs d'arrondi ; pour des données physiques, mettre le niveau de bruit)
    et `ceil_frac` x la taille des FLUCTUATIONS de Xa (pas sa valeur absolue :
    un décalage constant, ex. une aimantation moyenne, ne doit pas élargir la
    fenêtre). Protocole conseillé : écart initial ~1e-6 x cette taille, pour
    laisser plusieurs décades de mesure dans les deux sens.
    """
    d = np.linalg.norm(Xa - Xb, axis=1)
    ceil = ceil_frac * np.linalg.norm(Xa - Xa.mean(axis=0), axis=1).mean() + floor
    ok = (d > floor) & (d < ceil)                   # zone exponentielle
    idx = np.where(ok)[0]
    if len(idx) < 5:
        return np.nan
    idx = idx[: np.argmax(np.diff(idx) > 1) + 1] if np.any(np.diff(idx) > 1) else idx
    return np.polyfit(idx, np.log(d[idx]), 1)[0] if len(idx) >= 5 else np.nan


# ------------------------------------------------- 6. consistance
def consistency(Xa, Xc):
    """Corrélation moyenne, canal par canal, entre deux réponses à la MÊME entrée
    parties de départs VRAIMENT différents (transitoire déjà retiré).

    lambda ne teste qu'une perturbation infinitésimale. Un système multistable
    peut avoir lambda ~ 0 et ne jamais oublier son départ : seule la consistance
    le voit (ESN : 1.000, Coulomb à 25 charges : 0.0 à 0.35).
    """
    c = [np.corrcoef(a, b)[0, 1] for a, b in zip(Xa.T, Xc.T) if a.std() > 0 and b.std() > 0]
    return float(np.mean(c)) if c else np.nan


# ------------------------------------------------- le vecteur complet
def complexity_vector(X, Xb=None, D=5, warn=True, Xc=None):
    """Calcule le vecteur C sur des états X (T, N). Xb : 2e trajectoire optionnelle
    (petite perturbation, pour lambda). Xc : 3e trajectoire optionnelle (même entrée,
    départ vraiment différent, pour la consistance).

    Les états doivent être pris APRÈS le transitoire (le code ne le retire pas).
    """
    X = np.atleast_2d(X.T).T                         # accepte aussi une série 1D
    if Xb is not None:
        Xb = np.atleast_2d(Xb.T).T                   # même forme que X, sinon
        if Xb.shape != X.shape:                      # X - Xb « broadcast » en (T, T)
            raise ValueError(f"X {X.shape} et Xb {Xb.shape} doivent avoir la même forme")
    live = X.std(axis=0) > 0                         # canaux constants (nœud mort,
    if not live.all():                               # tanh saturé) : NaN sinon
        if warn:
            print(f"   ! attention : {np.sum(~live)} canal(aux) constant(s) retiré(s)")
        X = X[:, live]
        Xb = Xb[:, live] if Xb is not None else None
        Xc = Xc[:, live] if Xc is not None else None
    HC = np.array([entropy_complexity(x, D, warn=warn and i == 0) for i, x in enumerate(X.T)])
    tau_med, tau_sprd = timescales(X, warn=warn)
    cn, cn_form = neural_complexity(X) if X.shape[1] > 1 else (0.0, 0.0)
    return {
        "H_perm": HC[:, 0].mean(),
        "C_JS": HC[:, 1].mean(),
        "PR": participation_ratio(X) if X.shape[1] > 1 else 1.0,
        "C_N": cn,
        "C_N_form": cn_form,
        "tau_med": tau_med,
        "tau_sprd": tau_sprd,
        "lam": conditional_lyapunov(X, Xb) if Xb is not None else np.nan,
        "consist": consistency(X, np.atleast_2d(Xc.T).T) if Xc is not None else np.nan,
    }


# ================================================= validation par milestones
def self_test(T=20000, seed=1):
    rng = np.random.default_rng(seed)
    print("=" * 84)
    print("Étape 1 — validation du vecteur de complexité sur des signaux connus")
    print("=" * 84)

    # Signaux de référence
    noise = rng.standard_normal(T)
    sine = np.sin(2 * np.pi * np.arange(T) / 50)
    logi = np.empty(T); logi[0] = 0.4
    for t in range(T - 1):
        logi[t + 1] = 4 * logi[t] * (1 - logi[t])
    milestone("M1", True, "signaux générés : bruit blanc, sinus, carte logistique r=4")

    # M2-M3 : plan complexité-entropie (Rosso 2007)
    Hn, Cn = entropy_complexity(noise)
    Hs, Cs = entropy_complexity(sine)
    Hl, Cl = entropy_complexity(logi)
    print(f"   bruit    H={Hn:.3f}  C={Cn:.3f}")
    print(f"   sinus    H={Hs:.3f}  C={Cs:.3f}")
    print(f"   logist.  H={Hl:.3f}  C={Cl:.3f}")
    milestone("M2", Hn > 0.99 and Cn < 0.01, "bruit : entropie maximale, complexité ~0")
    milestone("M3", Cl > 5 * Cn and Hl < Hn, "chaos : plus complexe et moins entropique que le bruit")

    # M4 : dimension effective
    X_indep = rng.standard_normal((T, 20))
    X_sync = np.outer(sine, np.ones(20)) + 1e-3 * rng.standard_normal((T, 20))
    pr_i, pr_s = participation_ratio(X_indep), participation_ratio(X_sync)
    milestone("M4", pr_i > 0.9 and pr_s < 0.1,
              f"PR : indépendants={pr_i:.2f} (~1), synchronisés={pr_s:.2f} (~1/N)")

    # M5 : complexité neurale aux trois régimes (le régime synchronisé manquait)
    common = rng.standard_normal((T, 1))
    X_mix = 0.6 * common + 0.8 * rng.standard_normal((T, 20))
    (cn_i, f_i), (cn_m, f_m), (cn_s, f_s) = (neural_complexity(X) for X in (X_indep, X_mix, X_sync))
    print(f"   C_N brut : indépendants={cn_i:.3f}  intégrés+ségrégés={cn_m:.2f}  synchronisés={cn_s:.2f}")
    print(f"   forme    : indépendants={f_i:.3f}  intégrés+ségrégés={f_m:.3f}  synchronisés={f_s:.3f}")
    milestone("M5", cn_i < 0.05 and f_m > 2 * f_s and f_m > 2 * f_i,
              "forme de C_N : basse aux deux extrêmes, haute entre les deux")

    # M6 : temps de relaxation sur un AR(1), tau théorique = -1/ln(a)
    a, ar = 0.95, np.zeros((T, 1))
    for t in range(T - 1):
        ar[t + 1] = a * ar[t] + rng.standard_normal()
    tau_th = -1 / np.log(a)
    tau_med, _ = timescales(ar)
    milestone("M6", abs(tau_med - tau_th) < 0.25 * tau_th,
              f"tau AR(1) : mesuré={tau_med:.1f}, théorique={tau_th:.1f}")

    # M7-M8 : Lyapunov conditionnel, valeurs exactes connues
    xa, xb = np.empty(60), np.empty(60)
    xa[0], xb[0] = 0.4, 0.4 + 1e-9
    for t in range(59):
        xa[t + 1], xb[t + 1] = 4 * xa[t] * (1 - xa[t]), 4 * xb[t] * (1 - xb[t])
    lam_chaos = conditional_lyapunov(xa[:, None], xb[:, None], ceil_frac=1e-2)
    milestone("M7", abs(lam_chaos - np.log(2)) < 0.25,
              f"lambda logistique : {lam_chaos:.3f} (théorie ln2 = 0.693)")

    t = np.arange(60)
    ca, cb = (0.5 ** t)[:, None] * 1.0, (0.5 ** t)[:, None] * 2.0
    lam_stable = conditional_lyapunov(np.ones((60, 1)) + ca, np.ones((60, 1)) + cb)
    milestone("M8", abs(lam_stable - np.log(0.5)) < 0.05,
              f"lambda contractant : {lam_stable:.3f} (théorie ln0.5 = -0.693)")

    # M9 : le vecteur complet sur un cas multicanal
    C = complexity_vector(X_mix)
    finite = all(np.isfinite(v) for k, v in C.items() if k not in ("lam", "consist"))
    milestone("M9", finite, "vecteur complet calculé sans NaN")
    print("   C =", {k: round(float(v), 3) for k, v in C.items()})
    print("=" * 84)


if __name__ == "__main__":
    self_test()


## Step 2a — `reservoir.py`

leaky ESN, two-copy protocol

In [ ]:
"""
Étape 2a — Un réservoir minimal (ESN à fuite), la formule du notebook :

    r(t+1) = (1 - alpha) r(t) + alpha * tanh(W r(t) + W_in u(t) + b)

`step(r, u)` ne garde pas d'état interne : on peut donc faire avancer deux
copies (r_a et r_b) avec les MÊMES matrices et la MÊME entrée, ce qu'exige le
Lyapunov conditionnel.

Lancer `python reservoir.py` exécute les milestones R1 à R5.
"""

import time

import numpy as np


_T0 = time.perf_counter()


def milestone(tag, ok, msg):
    status = "OK  " if ok else "FAIL"
    print(f"[{tag}] [{status}] {msg:<62} ({time.perf_counter() - _T0:5.2f} s)")
    return ok


class ESN:
    def __init__(self, N=50, rho=0.9, alpha=0.3, input_scale=1.0, bias=0.1, seed=0):
        rng = np.random.default_rng(seed)
        W = rng.standard_normal((N, N))
        self.W = W * rho / np.max(np.abs(np.linalg.eigvals(W)))   # rayon spectral = rho
        self.W_in = rng.uniform(-1, 1, N) * input_scale
        self.b = rng.uniform(-1, 1, N) * bias
        self.N, self.rho, self.alpha = N, rho, alpha

    def step(self, r, u):
        """Un pas de temps : nouvel état à partir de l'état r et de l'entrée u."""
        return (1 - self.alpha) * r + self.alpha * np.tanh(self.W @ r + self.W_in * u + self.b)

    def run(self, u, r0=None):
        """Fait tourner le réservoir sur toute la séquence u. Renvoie X (T, N)."""
        r = np.zeros(self.N) if r0 is None else r0.copy()
        X = np.empty((len(u), self.N))
        for t, ut in enumerate(u):
            r = self.step(r, ut)
            X[t] = r
        return X


def lyapunov_pair(esn, u, delta=1e-6, warmup=500, seed=1):
    """Protocole du lambda conditionnel : on chauffe le réservoir, on crée une
    copie décalée de delta, puis on pilote les deux avec la même entrée."""
    r = esn.run(u[:warmup])[-1]                                   # fin du transitoire
    d0 = np.random.default_rng(seed).standard_normal(esn.N)
    Xa = esn.run(u[warmup:], r)
    Xb = esn.run(u[warmup:], r + delta * d0 / np.linalg.norm(d0))
    return conditional_lyapunov(Xa, Xb), Xa, Xb


def self_test():
    print("=" * 84)
    print("Étape 2a — validation du réservoir")
    print("=" * 84)
    rng = np.random.default_rng(0)
    u = rng.uniform(-1, 1, 3000)                                  # entrée i.i.d.

    esn = ESN(rho=0.9)
    rho_mes = np.max(np.abs(np.linalg.eigvals(esn.W)))
    milestone("R1", abs(rho_mes - 0.9) < 1e-9, f"rayon spectral mesuré = {rho_mes:.4f} (cible 0.9)")

    X = esn.run(u)
    milestone("R2", np.all(np.isfinite(X)) and np.abs(X).max() <= 1,
              f"3000 pas, états finis et dans [-1, 1] (max |r| = {np.abs(X).max():.2f})")

    # R3 : propriété d'écho, deux départs très différents finissent confondus
    Xa = esn.run(u, rng.uniform(-1, 1, esn.N))
    Xb = esn.run(u, rng.uniform(-1, 1, esn.N))
    gap = np.linalg.norm(Xa[-1] - Xb[-1])
    milestone("R3", gap < 1e-8, f"écho (rho=0.9) : écart final = {gap:.1e} (départs aléatoires)")

    # R4 : lambda conditionnel négatif en régime d'écho, positif en régime chaotique
    lam_echo, _, _ = lyapunov_pair(esn, u)
    chaos = ESN(rho=2.0, alpha=1.0, input_scale=0.05)
    lam_chaos, _, _ = lyapunov_pair(chaos, u)
    milestone("R4", lam_echo < 0 < lam_chaos,
              f"lambda : écho={lam_echo:+.3f}, rho=2 faible entrée={lam_chaos:+.3f}")

    # R5 : le vecteur de complexité sur les états du réservoir
    C = complexity_vector(X[500:], D=4)                           # transitoire retiré
    finite = all(np.isfinite(v) for k, v in C.items() if k not in ("lam", "consist"))
    milestone("R5", finite, "vecteur de complexité calculé sur les états")
    print("   C =", {k: round(float(v), 3) for k, v in C.items()})
    print("=" * 84)


if __name__ == "__main__":
    self_test()


## Step 3 — `substrates.py`

common substrate interface and technical check

In [ ]:
"""
Étape 3 — L'usine à réservoirs : une interface commune à tous les substrats.

Un substrat sait faire quatre choses (et une cinquième, optionnelle) :
    init_state(rng)          un état de départ
    step(s, u)               un pas de temps sous l'entrée u (sans modifier s)
    observe(s)               les canaux LUS, vecteur de taille n_obs (pas tout l'état)
    perturb(s, delta, rng)   une copie de s décalée d'une distance delta
    random_state(rng)        un départ vraiment différent (par défaut : décalé de 1)

Tout le reste est générique : run, run_pair, le vecteur de complexité, live_time.
`check_substrate` est le contrôle technique que tout nouveau substrat doit passer.

La cellule exécute les milestones S1 à S5 à la fin.
"""

import time

import numpy as np

# Notebook : pas d'import entre fichiers. Exécuter d'abord les cellules des étapes 1
# (complexity_vector, conditional_lyapunov) et 2a (ESN, lyapunov_pair).

_T0 = time.perf_counter()


def milestone(tag, ok, msg):
    status = "OK  " if ok else "FAIL"
    print(f"[{tag}] [{status}] {msg:<62} ({time.perf_counter() - _T0:5.2f} s)")
    return ok


# ------------------------------------------------------------- l'interface
class Substrate:
    n_obs = None                                       # nombre de canaux lus

    def init_state(self, rng=None):
        raise NotImplementedError

    def step(self, s, u):
        raise NotImplementedError

    def observe(self, s):
        raise NotImplementedError

    def random_state(self, rng):
        """Un départ VRAIMENT différent (pas une petite perturbation) : sert à tester
        la consistance et à évaluer la lecture sur un autre départ. Par défaut :
        l'état initial décalé d'une distance 1 ; chaque substrat peut faire mieux."""
        return self.perturb(self.init_state(rng), 1.0, rng)

    def perturb(self, s, delta, rng):
        """Par défaut : direction aléatoire, norme exactement delta."""
        d = rng.standard_normal(np.shape(s))
        return s + delta * d / np.linalg.norm(d)


def run(sub, u, s0=None, rng=None):
    """Fait tourner le substrat sur la séquence u. Renvoie (X (T, n_obs), état final)."""
    s = sub.init_state(rng) if s0 is None else s0
    X = np.empty((len(u), sub.n_obs))
    for t, ut in enumerate(u):
        s = sub.step(s, ut)
        X[t] = sub.observe(s)
    return X, s


def run_pair(sub, u, delta=1e-6, warmup=500, seed=1):
    """Deux copies, même entrée : on chauffe, on décale de delta, on fait tourner."""
    _, s = run(sub, u[:warmup])                        # fin du transitoire
    sb = sub.perturb(s, delta, np.random.default_rng(seed))
    Xa, _ = run(sub, u[warmup:], s)
    Xb, _ = run(sub, u[warmup:], sb)
    return Xa, Xb


# ------------------------------------------------------ le contrôle technique
def check_substrate(sub, T=300, seed=0):
    """Vérifie le contrat de l'interface. Renvoie la liste des problèmes (vide = OK)."""
    problems = []
    rng = np.random.default_rng(seed)
    u = rng.uniform(-1, 1, T)
    try:
        s0 = sub.init_state(np.random.default_rng(seed))
        o = np.asarray(sub.observe(s0))
        if o.shape != (sub.n_obs,):
            problems.append(f"observe renvoie {o.shape}, attendu ({sub.n_obs},)")
        s_copy = np.copy(s0)
        s1 = sub.step(s0, u[0])
        if not np.array_equal(s0, s_copy):
            problems.append("step modifie l'état reçu (il doit en renvoyer un nouveau)")
        if np.shape(s1) != np.shape(s0):
            problems.append(f"step change la forme de l'état {np.shape(s0)} -> {np.shape(s1)}")
        X1, _ = run(sub, u, rng=np.random.default_rng(seed))
        X2, _ = run(sub, u, rng=np.random.default_rng(seed))
        if not np.all(np.isfinite(X1)):
            problems.append(f"valeurs non finies dès le pas {np.argmax(~np.isfinite(X1).all(1))}")
        elif not np.array_equal(X1, X2):
            problems.append("non déterministe : même graine, même entrée, résultats différents")
        sr = np.copy(s_copy)                           # état neuf : step a pu abîmer s0
        sp = sub.perturb(sr, 1e-6, np.random.default_rng(seed))
        dist = np.linalg.norm(np.ravel(sp) - np.ravel(s_copy))
        if not np.isclose(dist, 1e-6, rtol=1e-3):
            problems.append(f"perturb décale de {dist:.2e} au lieu de 1e-6")
        if not np.array_equal(sr, s_copy):
            problems.append("perturb modifie l'état reçu")
    except Exception as e:                             # une erreur est aussi un problème
        problems.append(f"exception : {type(e).__name__}: {e}")
    return problems


# --------------------------------------------------------- premier substrat
class ESNSubstrate(Substrate):
    """L'ESN de reservoir.py derrière l'interface. obs_idx : nœuds lus (tous par défaut)."""

    def __init__(self, obs_idx=None, **kw):
        self.esn = ESN(**kw)
        self.obs_idx = np.arange(self.esn.N) if obs_idx is None else np.asarray(obs_idx)
        self.n_obs = len(self.obs_idx)

    def init_state(self, rng=None):
        return np.zeros(self.esn.N)

    def random_state(self, rng):
        return rng.uniform(-1, 1, self.esn.N)                   # tout l'intervalle de tanh

    def step(self, s, u):
        return self.esn.step(s, u)

    def observe(self, s):
        return s[self.obs_idx]


class _BrokenSubstrate(ESNSubstrate):
    """Volontairement cassé : sert à vérifier que le contrôle technique détecte les fautes."""

    def step(self, s, u):
        s[0] += 1e-3                                   # faute 1 : modifie l'état reçu
        return np.full_like(s, np.nan) if u > 0.9 else self.esn.step(s, u)    # faute 2 : NaN


# ================================================================= milestones
def self_test():
    print("=" * 84)
    print("Étape 3 — validation de l'interface des substrats")
    print("=" * 84)
    u = np.random.default_rng(0).uniform(-1, 1, 3000)
    sub = ESNSubstrate(rho=0.9)

    problems = check_substrate(sub)
    milestone("S1", not problems, "ESNSubstrate passe le contrôle technique")
    for p in problems:
        print("   -", p)

    X_old = sub.esn.run(u)
    X_new, _ = run(sub, u)
    milestone("S2", np.array_equal(X_old, X_new), "run() identique bit à bit à l'ancien ESN.run()")

    lam_old, Xa_old, Xb_old = lyapunov_pair(sub.esn, u)
    Xa, Xb = run_pair(sub, u)
    lam_new = conditional_lyapunov(Xa, Xb)
    same = np.array_equal(Xa, Xa_old) and np.array_equal(Xb, Xb_old) and lam_new == lam_old
    milestone("S3", same, f"run_pair() identique à lyapunov_pair() (lambda = {lam_new:+.4f})")

    part = ESNSubstrate(obs_idx=range(10), rho=0.9)
    Xp, _ = run(part, u)
    C = complexity_vector(Xp[500:], D=4, warn=False)
    ok = Xp.shape == (3000, 10) and np.array_equal(Xp, X_new[:, :10]) and np.isfinite(C["PR"])
    milestone("S4", ok, f"lecture partielle : 10 nœuds sur 50 lus, PR = {C['PR']:.3f}")

    problems = check_substrate(_BrokenSubstrate(rho=0.9))
    found = (any("step modifie" in p for p in problems) and any("non finies" in p for p in problems)
             and not any("perturb" in p for p in problems))
    milestone("S5", found, "substrat cassé : les 2 fautes trouvées, aucune fausse accusation")
    for p in problems:
        print("   -", p)
    print("=" * 84)


if __name__ == "__main__":
    self_test()


## Step 4 — `vector_space.py`

fixed normalisation, parallel coordinates

In [ ]:
"""
Étape 4 — L'espace des vecteurs : normalisation fixe et coordonnées parallèles.

Les axes sont normalisés par des bornes connues à l'avance (pas par une
population) : ils ne bougent pas pendant le direct, et deux systèmes mesurés à
des moments différents restent comparables.

Notebook : exécuter d'abord les cellules des étapes 1, 2a et 3.
La cellule exécute les milestones N1 à N5 à la fin.
"""

import time

import numpy as np

# Notebook : pas d'import entre fichiers (complexity_vector, ESNSubstrate, run
# viennent des cellules précédentes).

MAX_LAG = 500        # même borne que timescales()
TAU_SPRD_MAX = 2.0   # écart-type de log(tau) = 2 : échelles de temps étalées sur ~x50
LAM_SCALE = 0.1      # |lambda| = 0.1 / pas : déjà loin de la frontière écho/chaos

# (clé du vecteur, étiquette d'axe, ce que veulent dire 0 et 1)
AXES = [
    ("H_perm", "H\nentropie", "ordre → hasard"),
    ("C_JS", "C_JS\ncomplexité", "0 → 0.5"),
    ("PR", "PR\ndimension", "1 direction → toutes"),
    ("C_N_form", "C_N forme\nintrications", "aucune → max"),
    ("tau_med", "tau médian\n(log)", f"1 → {MAX_LAG} pas"),
    ("tau_sprd", "tau étalement", f"0 → {TAU_SPRD_MAX}"),
    ("lam", "lambda\n(milieu = 0)", "écho ← | → chaos"),
    ("consist", "consistance", "garde son départ → l'oublie"),
]

_T0 = time.perf_counter()


def milestone(tag, ok, msg):
    status = "OK  " if ok else "FAIL"
    print(f"[{tag}] [{status}] {msg:<62} ({time.perf_counter() - _T0:5.2f} s)")
    return ok


def normalize_vector(C):
    """Vecteur brut (dict) -> tableau de len(AXES) valeurs dans [0, 1] (NaN si absent)."""
    tau = max(C["tau_med"], 1.0)
    vals = {
        "H_perm": C["H_perm"],
        "C_JS": C["C_JS"] / 0.5,
        "PR": C["PR"],
        "C_N_form": C["C_N_form"],
        "tau_med": np.log(tau) / np.log(MAX_LAG),
        "tau_sprd": C["tau_sprd"] / TAU_SPRD_MAX,
        "lam": 0.5 + 0.5 * np.tanh(C["lam"] / LAM_SCALE),   # NaN reste NaN
        "consist": C.get("consist", np.nan),                 # corrélation : < 0 ramené à 0
    }
    return np.clip(np.array([vals[k] for k, _, _ in AXES], float), 0, 1)


def draw_axes(ax):
    """Dessine le cadre des coordonnées parallèles (axes verticaux, étiquettes)."""
    n = len(AXES)
    for i, (key, label, meaning) in enumerate(AXES):
        ax.axvline(i, color="0.75", lw=1, zorder=0)
        ax.annotate(meaning, xy=(i, 0), xycoords=("data", "axes fraction"),     # sous le nom
                    xytext=(0, -30), textcoords="offset points", ha="center", va="top",
                    fontsize=7, color="0.4")
    i_lam = [k for k, _, _ in AXES].index("lam")
    ax.plot([i_lam - 0.15, i_lam + 0.15], [0.5, 0.5], color="tab:red", lw=2)  # frontière écho/chaos
    ax.set_xticks(range(n))
    ax.set_xticklabels([label for _, label, _ in AXES], fontsize=8)
    ax.set(xlim=(-0.3, n - 0.7), ylim=(-0.02, 1.1), ylabel="valeur normalisée")


def plot_parallel(vectors, labels, ax=None, title="Coordonnées parallèles"):
    """Version statique : un dict brut par système, une ligne par système."""
    import matplotlib.pyplot as plt
    if ax is None:
        _, ax = plt.subplots(figsize=(11, 4.5))
    draw_axes(ax)
    for C, lab in zip(vectors, labels):
        ax.plot(range(len(AXES)), normalize_vector(C), "o-", lw=2, label=lab)
    ax.legend(fontsize=8, loc="upper left")
    ax.set_title(title)
    ax.figure.tight_layout()
    return ax


def self_test():
    print("=" * 84)
    print("Étape 4 — validation de la normalisation fixe")
    print("=" * 84)
    rng, T = np.random.default_rng(2), 20000

    X_noise = rng.standard_normal((T, 20))
    v = normalize_vector(complexity_vector(X_noise, warn=False) | {"lam": np.nan})
    milestone("N1", v[0] > 0.99 and v[1] < 0.01 and v[2] > 0.95,
              f"bruit blanc : H={v[0]:.2f} (1), C={v[1]:.2f} (0), PR={v[2]:.2f} (1)")

    lams = np.array([-1, -0.1, -0.01, 0, 0.01, 0.1, 1])
    base = complexity_vector(X_noise, warn=False)
    i_lam = [k for k, _, _ in AXES].index("lam")
    yl = np.array([normalize_vector(base | {"lam": l})[i_lam] for l in lams])
    milestone("N2", yl[3] == 0.5 and np.all(np.diff(yl) > 0),
              f"lambda=0 -> {yl[3]:.2f} (milieu), croissant de {yl[0]:.3f} à {yl[-1]:.3f}")

    t1 = normalize_vector(base | {"tau_med": 1.0})[4]
    tM = normalize_vector(base | {"tau_med": float(MAX_LAG)})[4]
    milestone("N3", t1 == 0 and tM == 1, f"tau : 1 pas -> {t1:.0f}, {MAX_LAG} pas -> {tM:.0f}")

    u = rng.uniform(-1, 1, 3000)
    systems = {"bruit blanc (20 canaux)": X_noise[:3000]}
    for rho in (0.5, 0.9, 1.2):
        Xa, Xb = run_pair(ESNSubstrate(rho=rho), u)
        systems[f"ESN rho={rho}"] = (Xa, Xb)
    vectors, labels = [], []
    for name, X in systems.items():
        C = (complexity_vector(X[0], X[1], D=4, warn=False) if isinstance(X, tuple)
             else complexity_vector(X, D=4, warn=False) | {"lam": np.nan})
        vectors.append(C); labels.append(name)
    norm = np.array([normalize_vector(C) for C in vectors])
    inside = np.all((norm >= 0) & (norm <= 1) | np.isnan(norm))
    milestone("N4", inside and norm[0, 3] < 0.05,
              "4 systèmes dans [0, 1], et bruit court : forme de C_N ~0")
    for name, row in zip(labels, norm):
        print(f"   {name:<24}" + " ".join(f"{x:5.2f}" for x in row))

    import matplotlib.pyplot as plt
    ax = plot_parallel(vectors, labels, title="Quatre systèmes dans l'espace des vecteurs")
    ax.figure.savefig("parallel_static.png", dpi=110)
    milestone("N5", True, "figure statique enregistrée : parallel_static.png")
    print("=" * 84)
    plt.show()


if __name__ == "__main__":
    self_test()


## Step 5 — `sweep.py`

benchmark (memory capacity, NARMA-10), ESN sweep

In [ ]:
"""
Étape 5 — Balayage de l'ESN : le vecteur suit-il la performance ?
(capacité mémoire et NARMA-10)

Grille rho x alpha, 3 graines par point, même entrée de référence (bruit i.i.d.).
Pour chaque réservoir : le vecteur de complexité (avec la consistance), la
capacité mémoire (MC) et l'erreur sur NARMA-10 (NRMSE). Banc honnête : la lecture
est apprise sur un départ et testée sur un autre départ, avec une autre entrée.

    [W..] milestones : le code marche (OK / FAIL)
    [RES] résultats  : ce que disent les données (ne peuvent pas « échouer »)

Notebook : exécuter d'abord les cellules des étapes 1, 2a, 3 et 4.
Puis : results = sweep()        (environ 30 s)
"""

import time

import numpy as np

# Notebook : pas d'import entre fichiers (complexity_vector, ESNSubstrate,
# run_pair, normalize_vector, AXES, draw_axes viennent des cellules précédentes).

RHOS = [0.2, 0.5, 0.8, 0.95, 1.1, 1.3]
ALPHAS = [0.1, 0.3, 0.6, 1.0]
SEEDS = [0, 1, 2]
T, WARMUP, K_MAX, NOISE = 3000, 500, 80, 0.01     # NOISE : même plancher que C_N

_T0 = time.perf_counter()


def milestone(tag, ok, msg):
    status = "OK  " if ok else "FAIL"
    print(f"[{tag}] [{status}] {msg:<62} ({time.perf_counter() - _T0:5.2f} s)")
    return ok


def result(msg):
    print(f"[RES] {msg}")


def _noisy(X, noise, rng):
    """Bruit de mesure : noise x écart-type de chaque canal."""
    return X + noise * X.std(axis=0) * rng.standard_normal(X.shape)


def memory_capacity(X, u, X_test=None, u_test=None, k_max=K_MAX, noise=NOISE, seed=0):
    """MC de Jaeger : somme sur k des r² pour retrouver u(t-k) avec une lecture linéaire.

    Les états reçoivent d'abord le bruit de mesure : sans lui, la lecture exploiterait
    des directions sous le plancher, celles que C_N ignore déjà.
    Sans X_test : apprentissage sur 70 % d'une même simulation, test sur les 30 %
    restants (ancien banc). Avec X_test : apprentissage sur X, test sur une AUTRE
    simulation (autre départ, autre entrée). C'est le banc honnête : une lecture qui
    n'a appris qu'une trajectoire échoue si le système n'oublie pas son départ.
    """
    rng = np.random.default_rng(seed)
    def design(Xs, us):
        A = np.hstack([_noisy(Xs, noise, rng)[k_max:], np.ones((len(Xs) - k_max, 1))])
        Y = np.stack([us[k_max - k: len(us) - k] for k in range(1, k_max + 1)], axis=1)
        return A, Y
    A, Y = design(X, u)
    if X_test is None:
        n_tr = int(0.7 * len(A))
        A, At, Y, Yt = A[:n_tr], A[n_tr:], Y[:n_tr], Y[n_tr:]
    else:
        At, Yt = design(X_test, u_test)
    W = np.linalg.lstsq(A, Y, rcond=None)[0]                          # tous les k d'un coup
    P = At @ W
    r2 = [np.corrcoef(P[:, k], Yt[:, k])[0, 1] ** 2 for k in range(k_max)]
    return float(np.sum(r2)), np.array(r2)


def narma10(u01):
    """Cible NARMA-10 (Atiya et Parlos 2000), entrée dans [0, 0.5] :
    y(t+1) = 0.3 y(t) + 0.05 y(t) Σ_{i=0..9} y(t-i) + 1.5 u(t-9) u(t) + 0.1
    Il faut à la fois de la mémoire (10 pas) et de la non-linéarité (produits)."""
    y = np.zeros(len(u01))
    for t in range(9, len(u01) - 1):
        y[t + 1] = (0.3 * y[t] + 0.05 * y[t] * y[t - 9: t + 1].sum()
                    + 1.5 * u01[t - 9] * u01[t] + 0.1)
    return y


def narma_nrmse(X, u, X_test=None, u_test=None, noise=NOISE, seed=0, skip=100):
    """NRMSE de la lecture linéaire sur NARMA-10 (0 = parfait, 1 = pas mieux qu'une constante).

    Plafonnée à 1 : au-delà, c'est un échec, et la valeur ne mesure plus que la force
    avec laquelle la lecture extrapole hors de sa zone d'apprentissage (le Coulomb
    donnait 28 à 141). Une régularisation ridge n'y change rien (testé : 52 à 141).

    Le réservoir reçoit u dans [-1, 1] ; la cible est calculée sur (u + 1) / 4, dans
    [0, 0.5] : même information, donc MC et NARMA se mesurent sur la même simulation.
    L'état X[t] a vu u(t) : il doit prédire y(t+1). X_test : comme memory_capacity.
    """
    rng = np.random.default_rng(seed)
    def design(Xs, us):
        y = narma10((us + 1) / 4)
        A = np.hstack([_noisy(Xs, noise, rng)[skip:-1], np.ones((len(Xs) - skip - 1, 1))])
        return A, y[skip + 1:]
    A, y = design(X, u)
    if X_test is None:
        n_tr = int(0.7 * len(A))
        A, At, y, yt = A[:n_tr], A[n_tr:], y[:n_tr], y[n_tr:]
    else:
        At, yt = design(X_test, u_test)
    w = np.linalg.lstsq(A, y, rcond=None)[0]
    err = At @ w - yt
    return float(min(np.sqrt(np.mean(err ** 2) / np.var(yt)), 1.0))


def evaluate(sub, seed=0, n_starts=3):
    """Le banc complet d'un substrat :
      A  départ standard, entrée de référence          -> vecteur, apprentissage
      B  A décalé de 1e-6, même entrée                 -> lambda
      C  n_starts autres départs, même entrée          -> consistance (moyenne)
      D  encore un autre départ, AUTRE entrée          -> test de la lecture
    Plusieurs départs pour la consistance : un seul peut tomber par chance dans la
    même configuration qu'A et cacher une multistabilité (vu sur un réseau de ressorts).
    Renvoie (vecteur brut, MC, NRMSE NARMA-10)."""
    u = np.random.default_rng(42).uniform(-1, 1, WARMUP + T)           # entrée de référence
    u_test = np.random.default_rng(7).uniform(-1, 1, WARMUP + T)       # entrée de test
    rng = np.random.default_rng(1000 + seed)
    Xa, Xb = run_pair(sub, u, warmup=WARMUP)
    C = complexity_vector(Xa, Xb, D=4, warn=False)
    C["consist"] = float(np.mean([consistency(Xa, run(sub, u, sub.random_state(rng))[0][WARMUP:])
                                  for _ in range(n_starts)]))
    Xd, _ = run(sub, u_test, sub.random_state(rng))
    mc = memory_capacity(Xa, u[WARMUP:], Xd[WARMUP:], u_test[WARMUP:])[0]
    nr = narma_nrmse(Xa, u[WARMUP:], Xd[WARMUP:], u_test[WARMUP:])
    return C, mc, nr


def _ranks(x):
    return np.argsort(np.argsort(x)).astype(float)


def spearman(x, y):
    ok = np.isfinite(x) & np.isfinite(y)
    return np.corrcoef(_ranks(x[ok]), _ranks(y[ok]))[0, 1]


def loco_knn_r2(F, y, groups, k=3):
    """R² de prédiction quand on retire TOUTES les graines d'un réglage à la fois :
    on prédit chaque réglage à partir des autres réglages seulement (k plus proches voisins)."""
    F = (F - np.nanmean(F, 0)) / (np.nanstd(F, 0) + 1e-12)
    F = np.nan_to_num(F)
    pred = np.empty_like(y)
    for g in np.unique(groups):
        test, train = groups == g, groups != g
        for i in np.where(test)[0]:
            d = np.linalg.norm(F[train] - F[i], axis=1)
            pred[i] = y[train][np.argsort(d)[:k]].mean()
    return 1 - np.sum((y - pred) ** 2) / np.sum((y - y.mean()) ** 2)


def analyze(name, perf, V, HP, groups, higher_is_better=True):
    """Corrélation de chaque axe avec la performance, puis prédiction d'un réglage jamais vu."""
    result(f"{name} : corrélation de rang (Spearman) entre chaque axe et la performance")
    sign = 1 if higher_is_better else -1                  # signe : + veut dire « aide »
    for j, (key, _, _) in enumerate(AXES):
        print(f"      {key:<9} {sign * spearman(V[:, j], perf):+.2f}")
    # Le résultat ne doit pas dépendre du nombre de voisins k : on en montre plusieurs.
    result(f"{name} d'un réglage jamais vu, R² selon le nombre de voisins k :")
    out = []
    for k in (1, 3, 5, 8):
        r2_vec, r2_hp = loco_knn_r2(V, perf, groups, k), loco_knn_r2(HP, perf, groups, k)
        out.append((r2_vec, r2_hp))
        print(f"      k={k}   vecteur {r2_vec:5.2f}   (rho, alpha) {r2_hp:5.2f}")
    return out


def sweep(show=True):
    print("=" * 84)
    print("Étape 5 — balayage de l'ESN : vecteur, capacité mémoire et NARMA-10")
    print("=" * 84)
    u = np.random.default_rng(42).uniform(-1, 1, WARMUP + T)            # entrée de référence
    y_check = np.concatenate([narma10((u + 1) / 4),
                              narma10((np.random.default_rng(7).uniform(-1, 1, WARMUP + T) + 1) / 4)])
    milestone("W0", np.all(np.isfinite(y_check)) and y_check.max() < 2,
              f"cible NARMA-10 stable (max {y_check.max():.2f})")
    rows = []                                    # (rho, alpha, seed, vecteur brut, MC, NRMSE)
    for rho in RHOS:
        for alpha in ALPHAS:
            for seed in SEEDS:
                C, mc, nr = evaluate(ESNSubstrate(rho=rho, alpha=alpha, seed=seed), seed)
                rows.append((rho, alpha, seed, C, mc, nr))
    n_cfg = len(RHOS) * len(ALPHAS)
    V = np.array([normalize_vector(r[3]) for r in rows])
    MC = np.array([r[4] for r in rows])
    NR = np.array([r[5] for r in rows])
    HP = np.array([[r[0], r[1]] for r in rows])
    groups = np.repeat(np.arange(n_cfg), len(SEEDS))
    milestone("W1", len(rows) == n_cfg * len(SEEDS) and np.all(np.isfinite(MC)),
              f"{len(rows)} réservoirs simulés ({n_cfg} réglages x {len(SEEDS)} graines)")
    N = 50                                       # canaux lus par l'ESN
    milestone("W2", np.all(MC <= N + 1e-9) and MC.max() > 2 * MC.min(),
              f"MC entre {MC.min():.1f} et {MC.max():.1f}, toujours <= N = {N}")
    milestone("W3", np.all(np.isfinite(NR)) and NR.min() > 0 and NR.max() < 1.5,
              f"NARMA-10 : NRMSE entre {NR.min():.2f} et {NR.max():.2f} (0 = parfait)")

    # Question 1 : l'écart entre graines est-il petit devant l'effet du réglage ?
    Vg = V.reshape(n_cfg, len(SEEDS), -1)
    within = np.nanmean(np.nanstd(Vg, axis=1), axis=0)
    between = np.nanstd(np.nanmean(Vg, axis=1), axis=0)
    result("signal/bruit par axe = écart entre réglages / écart entre graines")
    for (key, _, _), w, b in zip(AXES, within, between):
        print(f"      {key:<9} entre réglages {b:.3f}   entre graines {w:.3f}   rapport {b / w:6.1f}")
    for name, P in (("MC", MC), ("NRMSE", NR)):
        g = P.reshape(n_cfg, len(SEEDS))
        result(f"{name} : entre réglages {g.mean(1).std():.3f}, entre graines {g.std(1).mean():.3f}")

    # Question 2 : quels axes suivent chaque tâche, et le vecteur prédit-il ?
    analyze("MC", MC, V, HP, groups)
    analyze("NARMA-10", NR, V, HP, groups, higher_is_better=False)

    # Question 3 : les deux tâches veulent-elles la même dynamique ?
    mcg, nrg = MC.reshape(n_cfg, -1).mean(1), NR.reshape(n_cfg, -1).mean(1)
    cfg = HP[:: len(SEEDS)]
    b_mc, b_nr = cfg[np.argmax(mcg)], cfg[np.argmin(nrg)]
    result(f"meilleur réglage MC : rho={b_mc[0]}, alpha={b_mc[1]}   "
           f"meilleur NARMA : rho={b_nr[0]}, alpha={b_nr[1]}")
    result(f"accord entre les deux tâches (Spearman MC / -NRMSE par réglage) : "
           f"{spearman(mcg, -nrg):+.2f}")

    import matplotlib.pyplot as plt
    fig = plt.figure(figsize=(14, 13))
    gs = fig.add_gridspec(3, 3, height_ratios=[1, 1.05, 1.05])
    ax1, ax2, ax3 = (fig.add_subplot(gs[0, i]) for i in range(3))
    for ax, data, title, cmap in ((ax1, mcg, "MC (plus = mieux)", "viridis"),
                                  (ax2, nrg, "NARMA-10 NRMSE (moins = mieux)", "viridis_r")):
        im = ax.imshow(data.reshape(len(RHOS), len(ALPHAS)), origin="lower", aspect="auto", cmap=cmap)
        ax.set(xticks=range(len(ALPHAS)), xticklabels=ALPHAS, yticks=range(len(RHOS)),
               yticklabels=RHOS, xlabel="alpha", ylabel="rho", title=title)
        fig.colorbar(im, ax=ax)
    sc = ax3.scatter(MC, NR, c=HP[:, 1], cmap="plasma", s=18)
    ax3.set(xlabel="MC", ylabel="NRMSE NARMA-10", title="Les deux tâches (couleur = alpha)")
    fig.colorbar(sc, ax=ax3, label="alpha")

    for row, perf, label, cmap in ((1, MC, "MC", "viridis"), (2, NR, "NRMSE NARMA-10", "viridis_r")):
        axP = fig.add_subplot(gs[row, :])
        draw_axes(axP)
        norm = plt.Normalize(perf.min(), perf.max())
        order = np.argsort(perf if cmap == "viridis" else -perf)        # meilleurs au-dessus
        for i in order:
            axP.plot(range(len(AXES)), V[i], lw=1.2, alpha=0.6, color=plt.get_cmap(cmap)(norm(perf[i])))
        fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axP, label=label)
        axP.set_title(f"Les 72 réservoirs dans l'espace des vecteurs, colorés par {label} (clair = meilleur)")
    fig.tight_layout()
    fig.savefig("sweep.png", dpi=110)
    milestone("W4", True, "figure enregistrée : sweep.png")
    print("=" * 84)
    if show:
        plt.show()
    return {"rows": rows, "V": V, "MC": MC, "NR": NR, "HP": HP}


if __name__ == "__main__":
    results = sweep()


## Step 6 — `coulomb.py`

charges in a harmonic well (Coulomb)

In [ ]:
"""
Étape 6 — Deuxième substrat : N charges dans un puits harmonique (Coulomb).

Physique reprise du réservoir N-corps de Léo (github.com/lele394/
N-Body-electromagnetic-reservoir-computer), mêmes équations et mêmes valeurs :

    m a_i = Σ_j k q_i q_j (r_i - r_j) / (|r_i - r_j|² + ε²)^{3/2}
            - k_well r_i - γ v_i + q_i E0 u(t) ex

Différences voulues avec l'original (fait pour MNIST, donc statique) :
  - l'entrée arrive au fil du temps, par un champ électrique extérieur E0 u(t) ;
    les charges aléatoires (+ et -) font que le même champ pousse chaque particule
    différemment : c'est l'équivalent physique de W_in ;
  - on lit seulement positions et vitesses (ce qu'un capteur mesure), pas les
    énergies par particule, qui ajouteraient de la non-linéarité côté lecture ;
  - un pas d'entrée = n_sub pas d'intégration (Verlet-vitesse, dt = 0.04).

Notebook : exécuter d'abord les cellules des étapes 1, 2a, 3, 4 et 5.
La cellule exécute les milestones C1 à C5, puis : comp = compare()
"""

import time

import numpy as np

# Notebook : pas d'import entre fichiers (Substrate, check_substrate, run, run_pair,
# ESNSubstrate, complexity_vector, normalize_vector, draw_axes, AXES,
# memory_capacity, narma_nrmse, T, WARMUP viennent des cellules précédentes).

_T0 = time.perf_counter()


def milestone(tag, ok, msg):
    status = "OK  " if ok else "FAIL"
    print(f"[{tag}] [{status}] {msg:<62} ({time.perf_counter() - _T0:5.2f} s)")
    return ok


class CoulombSubstrate(Substrate):
    """État : vecteur plat [positions (N, 2), vitesses (N, 2)] de taille 4N."""

    def __init__(self, n=25, E0=0.5, k_coulomb=1.5, epsilon=0.4, k_well=0.2,
                 damping=0.05, dt=0.04, n_sub=5, seed=0):
        rng = np.random.default_rng(seed)
        self.n, self.E0, self.k, self.eps = n, E0, k_coulomb, epsilon
        self.k_well, self.gamma, self.dt, self.n_sub = k_well, damping, dt, n_sub
        self.q = rng.uniform(-1.0, 1.0, (n, 1))                 # charges fixes
        self.qq = self.q * self.q.T
        np.fill_diagonal(self.qq, 0.0)                           # pas d'auto-interaction
        self.pos0 = rng.uniform(-2, 2, (n, 2))                   # départ dans le puits
        self.n_obs = 4 * n

    def coulomb_force(self, pos):
        """Force de Coulomb adoucie sur chaque particule, (N, 2)."""
        diff = pos[:, None, :] - pos[None, :, :]                 # r_i - r_j
        inv_r3 = (np.sum(diff ** 2, axis=-1) + self.eps ** 2) ** -1.5
        return self.k * np.sum((self.qq * inv_r3)[:, :, None] * diff, axis=1)

    def accel(self, pos, vel, u):
        a = self.coulomb_force(pos) - self.k_well * pos - self.gamma * vel
        a[:, 0] += self.q[:, 0] * self.E0 * u                    # champ extérieur selon x
        return a                                                  # masses = 1

    def energy(self, s):
        """Énergie totale (cinétique + puits + Coulomb, paires comptées une fois)."""
        pos, vel = s[: 2 * self.n].reshape(-1, 2), s[2 * self.n:].reshape(-1, 2)
        diff = pos[:, None, :] - pos[None, :, :]
        r = np.sqrt(np.sum(diff ** 2, axis=-1) + self.eps ** 2)
        return (0.5 * np.sum(vel ** 2) + 0.5 * self.k_well * np.sum(pos ** 2)
                + 0.5 * np.sum(self.k * self.qq / r))

    def init_state(self, rng=None):
        return np.concatenate([self.pos0.ravel(), np.zeros(2 * self.n)])

    def random_state(self, rng):
        """Autres positions tirées dans le puits, au repos."""
        return np.concatenate([rng.uniform(-2, 2, 2 * self.n), np.zeros(2 * self.n)])

    def step(self, s, u):
        pos, vel = s[: 2 * self.n].reshape(-1, 2), s[2 * self.n:].reshape(-1, 2)
        acc = self.accel(pos, vel, u)
        for _ in range(self.n_sub):                              # Verlet-vitesse (comme Léo)
            v_half = vel + 0.5 * acc * self.dt
            pos = pos + v_half * self.dt
            acc = self.accel(pos, v_half, u)
            vel = v_half + 0.5 * acc * self.dt
        return np.concatenate([pos.ravel(), vel.ravel()])        # nouvel état, s intact

    def observe(self, s):
        return s


def self_test():
    print("=" * 84)
    print("Étape 6 — validation du substrat de Coulomb")
    print("=" * 84)
    sub = CoulombSubstrate()
    problems = check_substrate(sub)
    milestone("C1", not problems, f"contrôle technique ({sub.n} particules, {sub.n_obs} canaux)")
    for p in problems:
        print("   -", p)

    # C2 : mêmes accélérations que le code de Léo, si sa classe est chargée
    pos = np.random.default_rng(3).uniform(-2, 2, (sub.n, 2))
    vel = np.random.default_rng(4).normal(0, 0.3, (sub.n, 2))
    if "NBodyReservoir" in globals():
        ref = NBodyReservoir(n_particles=sub.n, dt=sub.dt)       # noqa: F821
        ref.charges, ref.damping = sub.q, sub.gamma
        err = np.abs(ref._accel(pos[None], vel[None])[0] - sub.accel(pos, vel, 0.0)).max()
        milestone("C2", err < 1e-12, f"accélérations identiques au code de Léo (écart max {err:.1e})")
    else:
        print("[C2] [----] code de Léo non chargé : comparaison sautée (optionnelle)")

    # C3 : 3e loi de Newton, les forces internes s'annulent deux à deux
    total = sub.coulomb_force(pos).sum(axis=0)
    milestone("C3", np.abs(total).max() < 1e-10, f"somme des forces de Coulomb = {np.abs(total).max():.1e} (~0)")

    # C4 : sans frottement ni entrée, l'énergie se conserve, avec une erreur en dt².
    # L'énergie totale (~1.5) est une petite différence de grands termes (cinétique ~10,
    # Coulomb négatif entre charges opposées) : on la compare à l'énergie cinétique.
    # Verlet est d'ordre 2 : diviser dt par 2 doit diviser l'erreur par ~4.
    spans = []
    for dt, n_sub in ((0.04, 5), (0.02, 10)):                     # même durée physique
        free = CoulombSubstrate(damping=0.0, E0=0.0, dt=dt, n_sub=n_sub)
        s, E, K = free.init_state(), [], []
        for _ in range(2000):
            s = free.step(s, 0.0)
            E.append(free.energy(s)); K.append(0.5 * np.sum(s[2 * free.n:] ** 2))
        spans.append((max(E) - min(E)) / np.mean(K))
    ratio = spans[0] / spans[1]
    milestone("C4", spans[0] < 1e-2 and 3 < ratio < 5,
              f"énergie : erreur {spans[0]:.1e} (relative à E_cin), /{ratio:.1f} si dt/2 (attendu /4)")

    # C5 : piloté par l'entrée de référence, le vecteur se calcule
    u = np.random.default_rng(42).uniform(-1, 1, WARMUP + T)
    Xa, Xb = run_pair(sub, u, warmup=WARMUP)
    C = complexity_vector(Xa, Xb, D=4, warn=False)
    finite = all(np.isfinite(v) for k, v in C.items() if k != "consist")   # pas de 3e trajectoire ici
    milestone("C5", finite, f"vecteur calculé, lambda = {C['lam']:+.4f}")
    print("   C =", {k: round(float(v), 3) for k, v in C.items()})
    print("=" * 84)


def compare(seeds=(0, 1, 2), show=True):
    """ESN et Coulomb sous la même entrée : vecteur, MC et NARMA-10, côte à côte."""
    print("=" * 84)
    print("Comparaison ESN / Coulomb (deux frottements) sous la même entrée de référence")
    print("=" * 84)
    u = np.random.default_rng(42).uniform(-1, 1, WARMUP + T)
    makers = {"ESN (rho=0.9, alpha=0.3)": lambda s: ESNSubstrate(seed=s),
              "Coulomb, gamma=0.05 (Léo)": lambda s: CoulombSubstrate(seed=s),
              "Coulomb, gamma=1.0": lambda s: CoulombSubstrate(damping=1.0, seed=s)}
    out = {}
    for name, make in makers.items():
        res = []
        for s in seeds:
            Xa, Xb = run_pair(make(s), u, warmup=WARMUP)
            C = complexity_vector(Xa, Xb, D=4, warn=False)
            res.append((C, memory_capacity(Xa, u[WARMUP:])[0], narma_nrmse(Xa, u[WARMUP:])))
        out[name] = res
        mc = np.array([r[1] for r in res]); nr = np.array([r[2] for r in res])
        print(f"{name:<26} MC = {mc.mean():5.2f} ± {mc.std():.2f}   "
              f"NARMA NRMSE = {nr.mean():.3f} ± {nr.std():.3f}")
    milestone("C6", all(np.isfinite(r[1]) and np.isfinite(r[2]) for v in out.values() for r in v),
              "les deux substrats mesurés sur les mêmes tâches")

    print("   vecteurs normalisés (moyenne des graines) :")
    print("   " + " " * 24 + " ".join(f"{k[:8]:>8}" for k, _, _ in AXES))
    for name, res in out.items():
        m = np.nanmean([normalize_vector(r[0]) for r in res], axis=0)
        print(f"   {name[:24]:<24}" + " ".join(f"{x:8.2f}" for x in m))

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(12, 4.8))
    draw_axes(ax)
    for (name, res), col in zip(out.items(), ("tab:blue", "tab:red", "tab:orange")):
        for i, r in enumerate(res):
            ax.plot(range(len(AXES)), normalize_vector(r[0]), "o-", color=col, lw=2, alpha=0.7,
                    label=f"{name}  MC={np.mean([x[1] for x in res]):.1f}, "
                          f"NARMA={np.mean([x[2] for x in res]):.2f}" if i == 0 else None)
    ax.legend(fontsize=8, loc="upper right")
    ax.set_title("ESN et Coulomb à deux frottements, dans l'espace des vecteurs (une ligne par graine)")
    fig.tight_layout()
    fig.savefig("compare_esn_coulomb.png", dpi=110)
    milestone("C7", True, "figure enregistrée : compare_esn_coulomb.png")
    print("=" * 84)
    if show:
        plt.show()
    return out


if __name__ == "__main__":
    self_test()


## Step 7 — `cross.py`

transfer test ESN -> Coulomb (about 10 min)

In [ ]:
"""
Étape 7 — Le test décisif : le vecteur prédit-il un substrat qu'on n'a jamais réglé ?

1. Balayage du Coulomb : frottement gamma x champ E0 x 3 graines (environ 6 min).
   On lit seulement les positions : 50 canaux, comme les 50 nœuds de l'ESN.
2. On apprend « vecteur -> performance » sur les 72 ESN de l'étape 5, puis on
   prédit les 72 Coulomb, et inversement. Les hyperparamètres ne peuvent pas
   faire ce test : rho et gamma n'ont rien en commun.

    [X..] milestones : le code marche (OK / FAIL)
    [RES] résultats  : ce que disent les données

Notebook : exécuter d'abord les cellules des étapes 1 à 6.
Puis : cross = cross_test()          (refait aussi le balayage ESN, ~10 s)
"""

import time

import numpy as np

# Notebook : pas d'import entre fichiers (CoulombSubstrate, run_pair, sweep,
# complexity_vector, normalize_vector, memory_capacity, narma_nrmse, AXES,
# T, WARMUP, SEEDS, spearman viennent des cellules précédentes).

GAMMAS = [0.05, 0.2, 0.5, 1.0, 2.0, 4.0]
E0S = [0.25, 0.5, 1.0, 2.0]

_T0 = time.perf_counter()


def milestone(tag, ok, msg):
    status = "OK  " if ok else "FAIL"
    print(f"[{tag}] [{status}] {msg:<62} ({time.perf_counter() - _T0:5.2f} s)")
    return ok


def result(msg):
    print(f"[RES] {msg}")


class CoulombPositions(CoulombSubstrate):
    """Même physique, mais on ne lit que les positions (2N = 50 canaux)."""

    def __init__(self, **kw):
        super().__init__(**kw)
        self.n_obs = 2 * self.n

    def observe(self, s):
        return s[: 2 * self.n]


def coulomb_sweep():
    rows = []
    for gamma in GAMMAS:
        for E0 in E0S:
            for seed in SEEDS:
                C, mc, nr = evaluate(CoulombPositions(damping=gamma, E0=E0, seed=seed), seed)
                rows.append((gamma, E0, seed, C, mc, nr))
        print(f"   gamma={gamma:<5} fait")
    return {"rows": rows, "V": np.array([normalize_vector(r[3]) for r in rows]),
            "MC": np.array([r[4] for r in rows]), "NR": np.array([r[5] for r in rows]),
            "HP": np.array([[r[0], r[1]] for r in rows])}


def _standardize(F_train, F_test):
    mu, sd = F_train.mean(0), F_train.std(0) + 1e-9
    return (F_train - mu) / sd, (F_test - mu) / sd


def knn_predict(F_train, y_train, F_test, k=5):
    A, B = _standardize(F_train, F_test)
    d = np.linalg.norm(B[:, None, :] - A[None, :, :], axis=2)
    return y_train[np.argsort(d, axis=1)[:, :k]].mean(axis=1)


def ridge_predict(F_train, y_train, F_test, alpha=1.0):
    A, B = _standardize(F_train, F_test)
    A1, B1 = np.hstack([A, np.ones((len(A), 1))]), np.hstack([B, np.ones((len(B), 1))])
    reg = alpha * np.eye(A1.shape[1]); reg[-1, -1] = 0
    w = np.linalg.solve(A1.T @ A1 + reg, A1.T @ y_train)
    return B1 @ w


def r2(y, p):
    """R² mesuré par rapport à la moyenne du substrat PRÉDIT : 0 = pas mieux que deviner
    sa moyenne (qu'on ne connaît pas), négatif = pire. Test sévère."""
    return 1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2)


def in_domain(F_train, F_test):
    """Un point est « dans le domaine » s'il est aussi près des points d'entraînement
    que ceux-ci le sont entre eux (95e centile des distances au plus proche voisin)."""
    A, B = _standardize(F_train, F_test)
    dAA = np.linalg.norm(A[:, None] - A[None], axis=2); np.fill_diagonal(dAA, np.inf)
    seuil = np.percentile(dAA.min(1), 95)
    return np.linalg.norm(B[:, None] - A[None], axis=2).min(1) <= seuil


def transfer(name_a, A, name_b, B):
    """Apprendre sur A, prédire B, pour MC et NARMA, avec plusieurs prédicteurs."""
    inside = in_domain(A["V"], B["V"])
    result(f"{name_a} -> {name_b} : {inside.sum()}/{len(inside)} points de {name_b} "
           f"dans le domaine couvert par {name_a}")
    lam = [j for j, (k, _, _) in enumerate(AXES) if k == "lam"]
    out = {}
    for task, key in (("MC", "MC"), ("NARMA", "NR")):
        ya, yb = A[key], B[key]
        preds = {"vecteur, k-NN": knn_predict(A["V"], ya, B["V"]),
                 "vecteur, linéaire": ridge_predict(A["V"], ya, B["V"]),
                 "lambda seul, k-NN": knn_predict(A["V"][:, lam], ya, B["V"][:, lam]),
                 "moyenne de A": np.full_like(yb, ya.mean())}
        print(f"      {task:<6} R² sur tout {name_b} | sur la partie dans le domaine")
        for pname, p in preds.items():
            inner = r2(yb[inside], p[inside]) if inside.sum() > 3 else np.nan
            print(f"        {pname:<20} {r2(yb, p):6.2f} | {inner:6.2f}")
        out[task] = (preds, inside)
    return out


def cross_test(esn=None, coul=None, show=True):
    print("=" * 84)
    print("Étape 7 — ESN et Coulomb : prédire un substrat jamais réglé")
    print("=" * 84)
    if esn is None:
        import contextlib, io
        with contextlib.redirect_stdout(io.StringIO()):                # étape 5, sans son texte
            esn = sweep(show=False)
    milestone("X1", len(esn["MC"]) == 72, "balayage ESN repris de l'étape 5 (72 réservoirs)")
    if coul is None:
        coul = coulomb_sweep()
    ok = np.all(np.isfinite(coul["MC"])) and np.all(np.isfinite(coul["NR"]))
    milestone("X2", ok and np.all(np.isfinite(coul["V"])),
              f"balayage Coulomb : {len(coul['MC'])} réservoirs, tout est fini")

    lam_c = np.array([r[3]["lam"] for r in coul["rows"]])
    result(f"Coulomb : {np.sum(lam_c > 0)}/{len(lam_c)} réservoirs chaotiques (lambda > 0)")
    result(f"Coulomb : MC de {coul['MC'].min():.1f} à {coul['MC'].max():.1f}, "
           f"NRMSE de {coul['NR'].min():.2f} à {coul['NR'].max():.2f}")
    result(f"ESN     : MC de {esn['MC'].min():.1f} à {esn['MC'].max():.1f}, "
           f"NRMSE de {esn['NR'].min():.2f} à {esn['NR'].max():.2f}")

    t_ec = transfer("ESN", esn, "Coulomb", coul)
    t_ce = transfer("Coulomb", coul, "ESN", esn)
    milestone("X3", True, "transferts calculés dans les deux sens")

    import matplotlib.pyplot as plt
    V = np.vstack([esn["V"], coul["V"]])
    Vc = V - V.mean(0)
    _, _, Wt = np.linalg.svd(Vc, full_matrices=False)                   # ACP commune
    P = Vc @ Wt[:2].T
    n_e = len(esn["V"])
    fig, axs = plt.subplots(2, 2, figsize=(13, 10))
    for ax, key, label, cmap in ((axs[0, 0], "MC", "MC", "viridis"),
                                 (axs[0, 1], "NR", "NRMSE NARMA (moins = mieux)", "viridis_r")):
        y = np.concatenate([esn[key], coul[key]])
        norm = plt.Normalize(y.min(), y.max())
        ax.scatter(P[:n_e, 0], P[:n_e, 1], c=y[:n_e], cmap=cmap, norm=norm, marker="o", s=30,
                   label="ESN", edgecolors="k", linewidths=0.3)
        sc = ax.scatter(P[n_e:, 0], P[n_e:, 1], c=y[n_e:], cmap=cmap, norm=norm, marker="^", s=40,
                        label="Coulomb", edgecolors="k", linewidths=0.3)
        fig.colorbar(sc, ax=ax, label=label)
        ax.set(xlabel="composante principale 1", ylabel="composante principale 2",
               title=f"Les deux substrats dans le même espace, couleur = {label}")
        ax.legend(fontsize=8)
    for ax, task, key in ((axs[1, 0], "MC", "MC"), (axs[1, 1], "NARMA", "NR")):
        preds, inside = t_ec[task]
        p, y = preds["vecteur, k-NN"], coul[key]
        ax.scatter(y[inside], p[inside], c="tab:green", s=25, label="dans le domaine de l'ESN")
        ax.scatter(y[~inside], p[~inside], c="tab:red", s=25, label="hors domaine")
        lo, hi = min(y.min(), p.min()), max(y.max(), p.max())
        ax.plot([lo, hi], [lo, hi], "k--", lw=1)
        ax.set(xlabel=f"{task} réelle du Coulomb", ylabel=f"{task} prédite (appris sur l'ESN)",
               title=f"{task} : prédire le Coulomb après avoir appris sur l'ESN")
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("cross_test.png", dpi=110)
    milestone("X4", True, "figure enregistrée : cross_test.png")
    print("=" * 84)
    if show:
        plt.show()
    return {"esn": esn, "coulomb": coul, "esn_to_coulomb": t_ec, "coulomb_to_esn": t_ce}


if __name__ == "__main__":
    cross = cross_test()


## Step 8 — `bench_check.py`

check of the corrected benchmark

In [ ]:
"""
Étape 8 — Vérifier le banc corrigé et l'axe de consistance.

Deux corrections, découvertes en comparant ESN et Coulomb :
  1. la lecture est apprise sur un départ et testée sur un AUTRE départ, avec une
     autre entrée (avant : 70 % / 30 % de la même simulation) ;
  2. le vecteur reçoit un 8e axe, la consistance : la même entrée, depuis deux
     départs vraiment différents, donne-t-elle la même réponse ?

Ce qu'on attend : rien ne change pour l'ESN (il oublie son départ), tout change
pour un système qui ne l'oublie pas (le Coulomb).

Notebook : exécuter d'abord les cellules des étapes 1 à 6 (sweep et coulomb
contiennent les nouvelles versions). La cellule exécute les milestones B1 à B6.
"""

import time

import numpy as np

# Notebook : pas d'import entre fichiers.

_T0 = time.perf_counter()


def milestone(tag, ok, msg):
    status = "OK  " if ok else "FAIL"
    print(f"[{tag}] [{status}] {msg:<62} ({time.perf_counter() - _T0:5.2f} s)")
    return ok


def result(msg):
    print(f"[RES] {msg}")


def old_bench(sub):
    """L'ancien banc, pour comparer : une seule simulation, 70 % / 30 %."""
    u = np.random.default_rng(42).uniform(-1, 1, WARMUP + T)
    Xa, _ = run_pair(sub, u, warmup=WARMUP)
    return memory_capacity(Xa, u[WARMUP:])[0], narma_nrmse(Xa, u[WARMUP:])


def self_test():
    print("=" * 84)
    print("Étape 8 — validation du banc corrigé et de l'axe de consistance")
    print("=" * 84)
    rng = np.random.default_rng(0)

    subs = {"ESN": ESNSubstrate(), "Coulomb": CoulombSubstrate(damping=1.0, E0=0.25)}
    ok = True
    for name, sub in subs.items():
        s0, s1 = sub.init_state(rng), sub.random_state(rng)
        ok &= np.shape(s1) == np.shape(s0) and np.linalg.norm(s1 - s0) > 0.5
        ok &= not check_substrate(sub)
    milestone("B1", ok, "random_state : forme correcte, départ vraiment différent")

    # B2 : la consistance sur des cas dont on connaît la réponse
    t = np.arange(3000)
    lin_a = np.outer(np.sin(t / 7), np.ones(5))
    lin_b = lin_a + 0.01 * rng.standard_normal(lin_a.shape)              # même réponse
    indep = rng.standard_normal(lin_a.shape)                            # rien en commun
    c_same, c_ind = consistency(lin_a, lin_b), consistency(lin_a, indep)
    milestone("B2", c_same > 0.99 and abs(c_ind) < 0.05,
              f"consistance : même réponse {c_same:.3f} (~1), indépendantes {c_ind:+.3f} (~0)")

    # B3 : ESN et Coulomb sous le banc complet
    res = {}
    for name, sub in subs.items():
        C, mc, nr = evaluate(sub)
        mc_old, nr_old = old_bench(sub)
        res[name] = (C, mc, nr, mc_old, nr_old)
        result(f"{name:<8} consistance {C['consist']:+.3f}   MC ancien {mc_old:5.2f} -> "
               f"nouveau {mc:5.2f}   NARMA ancien {nr_old:.3f} -> nouveau {nr:.3f}")
    milestone("B3", res["ESN"][0]["consist"] > 0.99,
              f"ESN : consistance {res['ESN'][0]['consist']:.4f} (> 0.99)")

    # B4 : pour un système consistant, le nouveau banc ne change presque rien
    C, mc, nr, mc_old, nr_old = res["ESN"]
    milestone("B4", abs(mc - mc_old) / mc_old < 0.15 and abs(nr - nr_old) < 0.05,
              f"ESN : MC {mc_old:.2f} -> {mc:.2f}, NARMA {nr_old:.3f} -> {nr:.3f} (inchangés)")

    # B5 : le vecteur a bien 8 axes, tous dans [0, 1]
    v = normalize_vector(res["Coulomb"][0])
    milestone("B5", len(v) == len(AXES) == 8 and np.all((v >= 0) & (v <= 1)),
              f"vecteur à {len(v)} axes, tous dans [0, 1]")
    milestone("B5b", res["Coulomb"][2] == 1.0,
              f"NRMSE plafonnée : échec du Coulomb affiché {res['Coulomb'][2]:.2f} (au lieu de ~28)")

    # B6 : la figure, ESN et Coulomb avec le nouvel axe
    import matplotlib.pyplot as plt
    ax = plot_parallel([res["ESN"][0], res["Coulomb"][0]],
                       [f"ESN  (MC {res['ESN'][1]:.1f}, NARMA {res['ESN'][2]:.2f})",
                        f"Coulomb gamma=1  (MC {res['Coulomb'][1]:.1f}, NARMA {res['Coulomb'][2]:.2f})"],
                       title="Le vecteur à 8 axes, banc corrigé")
    ax.figure.savefig("bench_check.png", dpi=110)
    milestone("B6", True, "figure enregistrée : bench_check.png")
    print("=" * 84)
    plt.show()


if __name__ == "__main__":
    self_test()


## Step 9 — `massspring.py`

mass-spring networks

In [ ]:
"""
Étape 9 — Troisième substrat : un réseau masses-ressorts (calcul morphologique).

Conception de Hauser et al. (2011) : des masses reliées à leurs voisines par des
ressorts amortis, quelques masses du bord fixées (ancrages), l'entrée appliquée
comme une force sur une partie des masses. Force d'un ressort d'allongement x :

    F = k1 x + k3 x³ + c dx/dt        (le long du ressort)

Deux versions qui ne diffèrent que par k3 :
    linéaire  : k3 = 0        cubique : k3 > 0
Attention : même « linéaire », le réseau n'est pas un système linéaire. La longueur
d'un ressort |r_i - r_j| dépend des positions de façon non linéaire (géométrie).
Il ne devient linéaire que pour de petites déformations.

Comme pour le Coulomb : 25 masses libres, on lit leurs positions (50 canaux),
un pas d'entrée = 0.2 unité de temps (4 pas de Verlet de 0.05).

Notebook : exécuter d'abord les cellules des étapes 1 à 8.
La cellule exécute les milestones K1 à K6, puis : prc = compare_prc()
"""

import time

import numpy as np

# Notebook : pas d'import entre fichiers.

_T0 = time.perf_counter()


def milestone(tag, ok, msg):
    status = "OK  " if ok else "FAIL"
    print(f"[{tag}] [{status}] {msg:<62} ({time.perf_counter() - _T0:5.2f} s)")
    return ok


def result(msg):
    print(f"[RES] {msg}")


class MassSpringSubstrate(Substrate):
    """État : vecteur plat [positions des masses libres (F, 2), vitesses (F, 2)]."""

    def __init__(self, n_free=25, n_anchor=6, k_neigh=4, k3=0.0, damping=1.0,
                 amp=0.3, input_frac=0.4, dt=0.05, n_sub=4, seed=0):
        # amp et damping : avec amp=1 et damping=0.3, le réseau se replie (ressorts écrasés
        # à 6 % de leur longueur) et devient multistable : consistance 0.32. Avec amp=0.3
        # et damping=1, il reste déformé sans se replier : consistance 1.000.
        rng = np.random.default_rng(seed)
        n = n_free + n_anchor
        P = rng.uniform(-2, 2, (n, 2))
        order = np.argsort(-np.linalg.norm(P, axis=1))                  # les plus extérieures
        self.anchor, self.free = order[:n_anchor], np.sort(order[n_anchor:])
        self.P0 = P                                                      # équilibre = départ
        D = np.linalg.norm(P[:, None] - P[None], axis=2)
        np.fill_diagonal(D, np.inf)
        edges = {tuple(sorted((i, j))) for i in range(n) for j in np.argsort(D[i])[:k_neigh]}
        self.ei, self.ej = np.array(sorted(edges)).T                     # ressorts (i, j)
        self.L0 = D[self.ei, self.ej]                                    # longueurs au repos
        self.k1 = rng.uniform(0.5, 1.5, len(self.L0))
        self.k3 = k3 * self.k1                                           # même rapport partout
        self.c = damping
        self.n, self.F, self.dt, self.n_sub = n, n_free, dt, n_sub
        w = np.zeros((n, 2))                                             # force d'entrée
        inp = rng.choice(self.free, int(input_frac * n_free), replace=False)
        w[inp] = rng.uniform(-1, 1, (len(inp), 2)) * amp
        self.w_in = w[self.free]
        self.n_obs = 2 * n_free

    # ------------------------------------------------------------- physique
    def _full(self, pos_free):
        P = self.P0.copy()
        P[self.free] = pos_free
        return P

    def spring_forces(self, P, V):
        """Force de chaque ressort sur toutes les masses (ancrages compris), (n, 2)."""
        d = P[self.ej] - P[self.ei]
        l = np.linalg.norm(d, axis=1)
        e = d / l[:, None]
        x = l - self.L0
        rate = np.sum((V[self.ej] - V[self.ei]) * e, axis=1)
        f = (self.k1 * x + self.k3 * x ** 3 + self.c * rate)[:, None] * e   # tire i vers j si étiré
        out = np.zeros_like(P)
        np.add.at(out, self.ei, f)
        np.add.at(out, self.ej, -f)
        return out

    def accel(self, pos, vel, u):
        V = np.zeros((self.n, 2)); V[self.free] = vel
        return self.spring_forces(self._full(pos), V)[self.free] + self.w_in * u   # masses = 1

    def energy(self, s):
        pos, vel = s[: 2 * self.F].reshape(-1, 2), s[2 * self.F:].reshape(-1, 2)
        P = self._full(pos)
        x = np.linalg.norm(P[self.ej] - P[self.ei], axis=1) - self.L0
        return 0.5 * np.sum(vel ** 2) + np.sum(0.5 * self.k1 * x ** 2 + 0.25 * self.k3 * x ** 4)

    # ------------------------------------------------------------- interface
    def init_state(self, rng=None):
        return np.concatenate([self.P0[self.free].ravel(), np.zeros(2 * self.F)])

    def random_state(self, rng):
        """Le même réseau, déformé : chaque masse libre déplacée jusqu'à ±0.3."""
        pos = self.P0[self.free] + rng.uniform(-0.3, 0.3, (self.F, 2))
        return np.concatenate([pos.ravel(), np.zeros(2 * self.F)])

    def step(self, s, u):
        pos, vel = s[: 2 * self.F].reshape(-1, 2), s[2 * self.F:].reshape(-1, 2)
        acc = self.accel(pos, vel, u)
        for _ in range(self.n_sub):                                      # Verlet-vitesse
            v_half = vel + 0.5 * acc * self.dt
            pos = pos + v_half * self.dt
            acc = self.accel(pos, v_half, u)
            vel = v_half + 0.5 * acc * self.dt
        return np.concatenate([pos.ravel(), vel.ravel()])

    def observe(self, s):
        return s[: 2 * self.F]


def self_test():
    print("=" * 84)
    print("Étape 9 — validation du réseau masses-ressorts")
    print("=" * 84)
    lin, cub = MassSpringSubstrate(k3=0.0), MassSpringSubstrate(k3=20.0)
    problems = check_substrate(lin) + check_substrate(cub)
    milestone("K1", not problems, f"contrôle technique ({lin.F} masses libres, "
                                  f"{len(lin.anchor)} ancrages, {len(lin.L0)} ressorts)")
    for p in problems:
        print("   -", p)

    # K2 : au repos et sans entrée, le réseau est à l'équilibre
    s0 = lin.init_state()
    a0 = np.abs(lin.accel(s0[:50].reshape(-1, 2), np.zeros((lin.F, 2)), 0.0)).max()
    milestone("K2", a0 < 1e-12, f"équilibre au départ : accélération max {a0:.1e} (~0)")

    # K3 : 3e loi de Newton (ancrages compris, les ressorts ne créent pas de force nette)
    rng = np.random.default_rng(1)
    P = lin.P0 + rng.normal(0, 0.2, lin.P0.shape)
    net = np.abs(lin.spring_forces(P, rng.normal(0, 0.3, P.shape)).sum(axis=0)).max()
    milestone("K3", net < 1e-12, f"somme des forces des ressorts = {net:.1e} (~0)")

    # K4 : sans frottement ni entrée, l'énergie se conserve : erreur en dt², pas de dérive.
    # Cas sévère exprès : ressorts cubiques (k3=20) et masses déplacées jusqu'à 0.3,
    # donc des ressorts jusqu'à ~6 fois plus raides qu'au repos.
    spans, drifts = [], []
    for dt, n_sub in ((0.05, 4), (0.025, 8)):
        free = MassSpringSubstrate(k3=20.0, damping=0.0, dt=dt, n_sub=n_sub)
        s = free.random_state(np.random.default_rng(2))
        E = []
        for _ in range(2000):
            s = free.step(s, 0.0)
            E.append(free.energy(s))
        E = np.array(E)
        spans.append((E.max() - E.min()) / E.mean())
        drifts.append(abs(E[-200:].mean() - E[:200].mean()) / E.mean())
    ratio = spans[0] / spans[1]
    milestone("K4", 3 < ratio < 5 and drifts[0] < spans[0],
              f"énergie : oscille de {spans[0]:.1e}, /{ratio:.1f} si dt/2 (attendu /4), "
              f"dérive {drifts[0]:.0e}")

    # K5 : le terme cubique n'agit que pour de grandes déformations
    u = np.random.default_rng(42).uniform(-1, 1, 1500)
    gaps = {}
    for amp in (0.01, 0.3):
        Xl, _ = run(MassSpringSubstrate(k3=0.0, amp=amp), u)
        Xc, _ = run(MassSpringSubstrate(k3=20.0, amp=amp), u)
        gaps[amp] = np.linalg.norm(Xl - Xc) / np.linalg.norm(Xl - Xl.mean(0))
    milestone("K5", gaps[0.01] < 0.01 and gaps[0.3] > 10 * gaps[0.01],
              f"écart linéaire/cubique : entrée 0.01 -> {gaps[0.01]:.4f}, entrée 0.3 -> {gaps[0.3]:.3f}")

    # K6 : banc complet sur les deux versions
    for name, sub in (("linéaire", lin), ("cubique", cub)):
        C, mc, nr = evaluate(sub)
        result(f"{name:<9} consistance {C['consist']:.3f}  lambda {C['lam']:+.4f}  "
               f"MC {mc:5.2f}  NARMA {nr:.3f}")
    milestone("K6", np.isfinite(mc) and np.isfinite(nr), "banc complet calculé sur les deux versions")
    print("=" * 84)


def compare_prc(seeds=(0, 1, 2), show=True):
    """Les deux PRC sous le même banc : masses-ressorts (2 versions) et Coulomb."""
    print("=" * 84)
    print("Comparaison des substrats physiques (banc honnête)")
    print("=" * 84)
    makers = {"Ressorts linéaires": lambda s: MassSpringSubstrate(k3=0.0, seed=s),
              "Ressorts cubiques": lambda s: MassSpringSubstrate(k3=20.0, seed=s),
              "Coulomb (gamma=1, E0=0.25)": lambda s: CoulombPositions(damping=1.0, E0=0.25, seed=s)}
    out = {}
    for name, make in makers.items():
        res = [evaluate(make(s), s) for s in seeds]
        out[name] = res
        mc = np.array([r[1] for r in res]); nr = np.array([r[2] for r in res])
        cs = np.array([r[0]["consist"] for r in res])
        print(f"{name:<27} consistance {cs.mean():.3f}   MC {mc.mean():5.2f} ± {mc.std():.2f}   "
              f"NARMA {np.median(nr):.3f} (médiane)")
    milestone("K7", all(np.isfinite(r[1]) for v in out.values() for r in v),
              "les trois substrats mesurés sous le même banc")

    print("   vecteurs normalisés (moyenne des graines) :")
    print("   " + " " * 27 + " ".join(f"{k[:8]:>8}" for k, _, _ in AXES))
    for name, res in out.items():
        m = np.nanmean([normalize_vector(r[0]) for r in res], axis=0)
        print(f"   {name[:27]:<27}" + " ".join(f"{x:8.2f}" for x in m))

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(13, 5))
    draw_axes(ax)
    for (name, res), col in zip(out.items(), ("tab:green", "tab:purple", "tab:orange")):
        for i, r in enumerate(res):
            lab = (f"{name}  MC={np.mean([x[1] for x in res]):.1f}, "
                   f"NARMA={np.median([x[2] for x in res]):.2f}") if i == 0 else None
            ax.plot(range(len(AXES)), normalize_vector(r[0]), "o-", color=col, lw=2, alpha=0.7, label=lab)
    ax.legend(fontsize=8, loc="upper right")
    ax.set_title("Trois substrats physiques dans l'espace des vecteurs (une ligne par graine)")
    fig.tight_layout()
    fig.savefig("compare_prc.png", dpi=110)
    milestone("K8", True, "figure enregistrée : compare_prc.png")
    print("=" * 84)
    if show:
        plt.show()
    return out


if __name__ == "__main__":
    self_test()


## Step 10 — `geometry.py`

geometries as hidden parameters (about 5 min)

In [ ]:
"""
Étape 10 — Les géométries comme paramètres cachés de fabrication.

Chaque graine tire un réseau masses-ressorts différent (positions, ressorts,
masses qui reçoivent l'entrée), comme deux puces « identiques » d'un même lot.
Question : SANS connaître la géométrie, le vecteur dit-il quels réseaux marchent ?

Grille : 2 amplitudes d'entrée x 2 amortissements (le point de fonctionnement,
connu et réglable) x 12 géométries (cachées) = 48 réseaux, environ 6 minutes.

Test : on retire une géométrie entière (ses 4 points de fonctionnement), on la
prédit à partir des 11 autres. Prédicteurs comparés :
  - le point de fonctionnement seul (amp, amortissement) : ne voit pas la géométrie
  - la consistance seule
  - le vecteur sans la consistance (la « richesse » seule)
  - le vecteur complet

Notebook : exécuter d'abord les cellules des étapes 1 à 9.
Puis : geo = geometry_test()
"""

import time

import numpy as np

# Notebook : pas d'import entre fichiers.

AMPS = [0.15, 0.3]
DAMPINGS = [0.5, 1.0]
GEOMETRIES = list(range(12))

_T0 = time.perf_counter()


def milestone(tag, ok, msg):
    status = "OK  " if ok else "FAIL"
    print(f"[{tag}] [{status}] {msg:<62} ({time.perf_counter() - _T0:5.2f} s)")
    return ok


def result(msg):
    print(f"[RES] {msg}")


def geometry_sweep():
    rows = []                                    # (amp, damping, géométrie, vecteur, MC, NRMSE)
    for g in GEOMETRIES:
        for amp in AMPS:
            for c in DAMPINGS:
                C, mc, nr = evaluate(MassSpringSubstrate(amp=amp, damping=c, seed=g), g)
                rows.append((amp, c, g, C, mc, nr))
        print(f"   géométrie {g:2d} faite   (MC aux 4 points : "
              + " ".join(f"{r[4]:4.1f}" for r in rows[-4:]) + ")")
    return {"rows": rows,
            "V": np.array([normalize_vector(r[3]) for r in rows]),
            "MC": np.array([r[4] for r in rows]),
            "NR": np.array([r[5] for r in rows]),
            "OP": np.array([[r[0], r[1]] for r in rows]),
            "G": np.array([r[2] for r in rows])}


def geometry_test(data=None, show=True):
    print("=" * 84)
    print("Étape 10 — le vecteur reconnaît-il les bonnes géométries ?")
    print("=" * 84)
    if data is None:
        data = geometry_sweep()
    V, MC, NR, OP, G = data["V"], data["MC"], data["NR"], data["OP"], data["G"]
    milestone("G1", len(MC) == len(GEOMETRIES) * len(AMPS) * len(DAMPINGS)
              and np.all(np.isfinite(V)), f"{len(MC)} réseaux mesurés, vecteurs complets")

    keys = [k for k, _, _ in AXES]
    ic = keys.index("consist")
    result("par point de fonctionnement (les 12 géométries) :")
    for amp in AMPS:
        for c in DAMPINGS:
            m = (OP[:, 0] == amp) & (OP[:, 1] == c)
            print(f"      amp={amp:<5} amort.={c:<4} MC {MC[m].mean():5.2f} ± {MC[m].std():.2f}   "
                  f"NARMA {NR[m].mean():.2f} ± {NR[m].std():.2f}   "
                  f"consistants (>0.95) : {np.sum(V[m, ic] > 0.95)}/{m.sum()}")

    # Combien de la variation vient de la géométrie, combien du point de fonctionnement ?
    for name, y in (("MC", MC), ("NARMA", NR)):
        op_means = np.array([y[(OP[:, 0] == a) & (OP[:, 1] == c)].mean() for a, c in OP])
        share = 1 - np.var(y - op_means) / np.var(y)
        result(f"{name} : le point de fonctionnement explique {share:.0%} de la variation, "
               f"la géométrie cachée le reste ({1 - share:.0%})")

    result("corrélation de rang avec la performance (+ = aide), sur les 48 réseaux")
    print("      " + " ".join(f"{k[:8]:>9}" for k in keys))
    print("  MC  " + " ".join(f"{spearman(V[:, j], MC):+9.2f}" for j in range(len(keys))))
    print("NARMA " + " ".join(f"{-spearman(V[:, j], NR):+9.2f}" for j in range(len(keys))))

    feats = {"point de fonctionnement": OP,
             "consistance seule": V[:, [ic]],
             "vecteur sans consistance": np.delete(V, ic, axis=1),
             "vecteur complet": V}
    result("prédire une géométrie jamais vue (R², selon le nombre de voisins k)")
    scores = {}
    for task, y in (("MC", MC), ("NARMA", NR)):
        print(f"      {task:<6}" + "".join(f"{'k=' + str(k):>8}" for k in (3, 5, 8)))
        for name, F in feats.items():
            r = [loco_knn_r2(F, y, G, k) for k in (3, 5, 8)]
            scores[(task, name)] = r
            print(f"        {name:<25}" + "".join(f"{x:8.2f}" for x in r))
    milestone("G2", all(np.all(np.isfinite(v)) for v in scores.values()), "prédictions calculées")

    import matplotlib.pyplot as plt
    fig = plt.figure(figsize=(14, 10))
    gs = fig.add_gridspec(2, 3, height_ratios=[1, 1.05])
    ax1, ax2, ax3 = (fig.add_subplot(gs[0, i]) for i in range(3))
    labels = {(a, c): f"amp {a}, amort. {c}" for a in AMPS for c in DAMPINGS}
    for (a, c), lab in labels.items():
        m = (OP[:, 0] == a) & (OP[:, 1] == c)
        ax1.scatter(V[m, ic], MC[m], s=25, label=lab)
        ax2.scatter(V[m, ic], NR[m], s=25, label=lab)
    ax1.set(xlabel="consistance", ylabel="MC", title="MC contre consistance")
    ax2.set(xlabel="consistance", ylabel="NRMSE NARMA (plafonnée à 1)", title="NARMA contre consistance")
    ax1.legend(fontsize=7); ax2.legend(fontsize=7)

    F = V
    pred = np.empty_like(MC)
    for g in np.unique(G):
        te, tr = G == g, G != g
        A, B = (F[tr] - F[tr].mean(0)) / (F[tr].std(0) + 1e-9), (F[te] - F[tr].mean(0)) / (F[tr].std(0) + 1e-9)
        d = np.linalg.norm(B[:, None] - A[None], axis=2)
        pred[te] = MC[tr][np.argsort(d, axis=1)[:, :5]].mean(1)
    ax3.scatter(MC, pred, c=V[:, ic], cmap="RdYlGn", s=25, vmin=0, vmax=1)
    lo, hi = min(MC.min(), pred.min()), max(MC.max(), pred.max())
    ax3.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax3.set(xlabel="MC réelle", ylabel="MC prédite (géométrie jamais vue, k=5)",
            title="Prédiction (vecteur, k=5)\ncouleur = consistance")

    axP = fig.add_subplot(gs[1, :])
    draw_axes(axP)
    norm = plt.Normalize(MC.min(), MC.max())
    for i in np.argsort(MC):
        axP.plot(range(len(AXES)), V[i], lw=1.2, alpha=0.6, color=plt.cm.viridis(norm(MC[i])))
    fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap="viridis"), ax=axP, label="MC")
    axP.set_title("48 réseaux masses-ressorts (12 géométries x 4 points), colorés par MC")
    fig.tight_layout()
    fig.savefig("geometry_test.png", dpi=110)
    milestone("G3", True, "figure enregistrée : geometry_test.png")
    print("=" * 84)
    if show:
        plt.show()
    return data | {"scores": scores}


if __name__ == "__main__":
    geo = geometry_test()


## Step 11 — `forecast.py`

forecasting benchmark, linear solvability rule

In [ ]:
"""
Étape 11 — Le banc de prévision, et le test de solubilité linéaire.

Tâche : à l'instant t, le réservoir a reçu la série s jusqu'à s(t) ; il doit
prédire s(t + h), h pas plus loin.

Deux séries chaotiques connues :
  Mackey-Glass (tau = 17), échantillonnée toutes les 1 unité de temps
  Lorenz (composante x), échantillonnée toutes les 0.05 unité de temps

Règle du banc : on ne garde que les tâches (série, horizon h) qu'un modèle
LINÉAIRE ne sait pas résoudre, mais qu'un modèle NON LINÉAIRE simple sait
améliorer. Sinon, un réservoir qui réussit ne prouverait rien sur la non-linéarité
(leçon d'ENSO). Les références :
  persistance   s(t+h) = s(t)                    (ne rien savoir)
  AR linéaire   combinaison des 50 dernières valeurs (même nombre de canaux que le réservoir)
  NG-RC         8 dernières valeurs + tous leurs produits deux à deux (44 canaux)

Même honnêteté que le banc précédent : apprentissage sur un morceau de la
série, test sur un AUTRE morceau, depuis un autre départ ; bruit de mesure de 1 %.

Notebook : exécuter d'abord les cellules des étapes 1 à 10.
La cellule exécute les milestones F1 à F3, puis : fc = forecast_bench()
"""

import time

import numpy as np

# Notebook : pas d'import entre fichiers.

HORIZONS = [1, 5, 10, 20]
AR_ORDER, NG_DELAYS = 50, 8
SEG_TRAIN, SEG_TEST = (0, 3500), (4500, 8000)          # deux morceaux disjoints
AR_FAIL, NL_GAIN = 0.2, 0.7      # gardée si AR > 0.2 ET NG-RC < 0.7 x AR (fixés avant tout réservoir)

_T0 = time.perf_counter()


def milestone(tag, ok, msg):
    status = "OK  " if ok else "FAIL"
    print(f"[{tag}] [{status}] {msg:<62} ({time.perf_counter() - _T0:5.2f} s)")
    return ok


def result(msg):
    print(f"[RES] {msg}")


# ------------------------------------------------------------------ les séries
def mackey_glass(n=8000, tau=17.0, dt=0.05, sample=1.0, discard=1000, seed=0):
    """dx/dt = 0.2 x(t-tau) / (1 + x(t-tau)^10) - 0.1 x   (Euler, pas dt)."""
    rng = np.random.default_rng(seed)
    d, k = int(round(tau / dt)), int(round(sample / dt))
    total = (n + discard) * k + d
    x = np.empty(total)
    x[: d + 1] = 1.2 + 0.01 * rng.standard_normal(d + 1)          # histoire initiale
    for t in range(d, total - 1):
        xd = x[t - d]
        x[t + 1] = x[t] + dt * (0.2 * xd / (1 + xd ** 10) - 0.1 * x[t])
    return x[d::k][discard: discard + n]


def lorenz(n=8000, dt=0.01, sample=0.05, discard=1000, x0=(1.0, 1.0, 1.0), full=False):
    """Lorenz (sigma=10, rho=28, beta=8/3), Runge-Kutta 4. Renvoie la composante x."""
    def f(v):
        x, y, z = v
        return np.array([10 * (y - x), x * (28 - z) - y, x * y - 8 / 3 * z])
    k = int(round(sample / dt))
    v, out = np.array(x0, float), []
    for i in range((n + discard) * k):
        k1 = f(v); k2 = f(v + dt / 2 * k1); k3 = f(v + dt / 2 * k2); k4 = f(v + dt * k3)
        v = v + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        if i % k == k - 1:
            out.append(v.copy())
    out = np.array(out)[discard:]
    return out if full else out[:, 0]


def normalize(s):
    """Ramène la série dans [-1, 1] avec les bornes du morceau d'apprentissage."""
    a, b = s[SEG_TRAIN[0]: SEG_TRAIN[1]].min(), s[SEG_TRAIN[0]: SEG_TRAIN[1]].max()
    return 2 * (s - a) / (b - a) - 1


# ------------------------------------------------------------- les références
def _nrmse(p, y):
    return float(min(np.sqrt(np.mean((p - y) ** 2) / np.var(y)), 1.0))


def _fit_predict(F_tr, y_tr, F_te, noise=NOISE, seed=0):
    rng = np.random.default_rng(seed)
    A = np.hstack([_noisy(F_tr, noise, rng), np.ones((len(F_tr), 1))])
    B = np.hstack([_noisy(F_te, noise, rng), np.ones((len(F_te), 1))])
    w = np.linalg.lstsq(A, y_tr, rcond=None)[0]
    return B @ w


def _delays(s, start, stop, p):
    """Lignes t = start..stop-1 : [s(t), s(t-1), ..., s(t-p+1)]."""
    return np.stack([s[start - j: stop - j] for j in range(p)], axis=1)


def baselines(s, h, warm=WARMUP):
    """NRMSE de la persistance, de l'AR linéaire et du NG-RC quadratique."""
    out = {}
    (a0, a1), (b0, b1) = SEG_TRAIN, SEG_TEST
    t_tr, t_te = (a0 + warm, a1 - h), (b0 + warm, b1 - h)
    y_tr, y_te = s[t_tr[0] + h: t_tr[1] + h], s[t_te[0] + h: t_te[1] + h]
    out["persistance"] = _nrmse(s[t_te[0]: t_te[1]], y_te)
    out["AR linéaire"] = _nrmse(_fit_predict(_delays(s, *t_tr, AR_ORDER), y_tr,
                                             _delays(s, *t_te, AR_ORDER)), y_te)
    def ng(t0, t1):
        L = _delays(s, t0, t1, NG_DELAYS)
        iu = np.triu_indices(NG_DELAYS)
        return np.hstack([L, (L[:, :, None] * L[:, None, :])[:, iu[0], iu[1]]])
    out["NG-RC"] = _nrmse(_fit_predict(ng(*t_tr), y_tr, ng(*t_te)), y_te)
    return out


def forecast_nrmse(sub, s, h, seed=0, warm=WARMUP):
    """Le réservoir piloté par la série : appris sur un morceau depuis le départ
    standard, testé sur un autre morceau depuis un autre départ."""
    (a0, a1), (b0, b1) = SEG_TRAIN, SEG_TEST
    X_tr, _ = run(sub, s[a0:a1])
    X_te, _ = run(sub, s[b0:b1], sub.random_state(np.random.default_rng(2000 + seed)))
    F_tr, y_tr = X_tr[warm: len(X_tr) - h], s[a0 + warm + h: a1]
    F_te, y_te = X_te[warm: len(X_te) - h], s[b0 + warm + h: b1]
    return _nrmse(_fit_predict(F_tr, y_tr, F_te, seed=seed), y_te)


# ================================================================= milestones
def self_test():
    print("=" * 84)
    print("Étape 11 — validation des séries chaotiques")
    print("=" * 84)
    mg = mackey_glass()
    ok = 0.2 < mg.min() and mg.max() < 1.5 and np.std(mg) > 0.1
    milestone("F1", ok, f"Mackey-Glass : valeurs entre {mg.min():.2f} et {mg.max():.2f} (attendu ~0.4-1.3)")

    # F2 : l'exposant de Lyapunov de Lorenz est connu, ~0.906 par unité de temps.
    # Méthode de Benettin : on renormalise l'écart à chaque pas et on moyenne sa croissance
    # sur une longue trajectoire. (Une seule pente sur une trajectoire courte le sous-estime :
    # 0.32 sur 12 unités de temps, car l'étirement de Lorenz est très inégal sur l'attracteur.
    # Limite à garder en tête pour notre conditional_lyapunov, qui mesure une seule pente.)
    def f(v):
        x, y, z = v
        return np.array([10 * (y - x), x * (28 - z) - y, x * y - 8 / 3 * z])
    def rk4(v, dt=0.01):
        k1 = f(v); k2 = f(v + dt / 2 * k1); k3 = f(v + dt / 2 * k2); k4 = f(v + dt * k3)
        return v + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    v = np.array([1.0, 1.0, 1.0])
    for _ in range(2000):
        v = rk4(v)
    w, acc, n_steps = v + np.array([1e-8, 0, 0]), 0.0, 20000
    for _ in range(n_steps):
        v, w = rk4(v), rk4(w)
        d = np.linalg.norm(w - v)
        acc += np.log(d / 1e-8)
        w = v + (w - v) * 1e-8 / d
    lam = acc / (n_steps * 0.01)
    milestone("F2", abs(lam - 0.906) < 0.1, f"Lorenz : lambda = {lam:.3f} par unité de temps (théorie 0.906)")

    # F3 : les deux morceaux de série sont bien disjoints et de même nature
    s = normalize(mg)
    same = abs(s[SEG_TRAIN[0]:SEG_TRAIN[1]].std() - s[SEG_TEST[0]:SEG_TEST[1]].std()) < 0.1
    milestone("F3", SEG_TRAIN[1] < SEG_TEST[0] and same,
              "apprentissage et test : morceaux disjoints, même statistique")
    print("=" * 84)


def forecast_bench(show=True):
    print("=" * 84)
    print("Le banc de prévision : quelles tâches sont vraiment non linéaires ?")
    print("=" * 84)
    series = {"Mackey-Glass": normalize(mackey_glass()), "Lorenz x": normalize(lorenz())}
    table, kept = {}, []
    for name, s in series.items():
        print(f"   {name}")
        print(f"      {'h':>3} {'persistance':>12} {'AR linéaire':>12} {'NG-RC':>8} {'ESN':>8}   gardée ?")
        for h in HORIZONS:
            b = baselines(s, h)
            esn = np.mean([forecast_nrmse(ESNSubstrate(seed=g), s, h, g) for g in range(3)])
            keep = b["AR linéaire"] > AR_FAIL and b["NG-RC"] < NL_GAIN * b["AR linéaire"]
            table[(name, h)] = b | {"ESN": esn, "gardée": keep}
            if keep:
                kept.append((name, h))
            print(f"      {h:>3} {b['persistance']:12.3f} {b['AR linéaire']:12.3f} "
                  f"{b['NG-RC']:8.3f} {esn:8.3f}   {'OUI' if keep else 'non'}")
    milestone("F4", len(kept) > 0 and all(np.isfinite(v["ESN"]) for v in table.values()),
              f"{len(kept)} tâches gardées sur {len(table)}")
    result("tâches retenues : " + ", ".join(f"{n} h={h}" for n, h in kept))

    import matplotlib.pyplot as plt
    fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
    for ax, name in zip(axs, series):
        for key, style in (("persistance", ":"), ("AR linéaire", "--"), ("NG-RC", "-."), ("ESN", "-")):
            ax.plot(HORIZONS, [table[(name, h)][key] for h in HORIZONS], style, marker="o", label=key)
        for h in HORIZONS:
            if table[(name, h)]["gardée"]:
                ax.axvspan(h - 0.6, h + 0.6, color="tab:green", alpha=0.12)
        ax.axhline(AR_FAIL, color="0.6", lw=0.8)
        ax.set(xlabel="horizon h (pas)", ylabel="NRMSE (plafonnée à 1)", title=f"{name} (vert = tâche gardée)",
               xticks=HORIZONS, ylim=(0, 1.05))
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("forecast_bench.png", dpi=110)
    milestone("F5", True, "figure enregistrée : forecast_bench.png")
    print("=" * 84)
    if show:
        plt.show()
    return {"series": series, "table": table, "kept": kept}


if __name__ == "__main__":
    self_test()


## Step 12 — `structure.py`

structural filter Q1-Q4

In [ ]:
"""
Étape 12 — Le filtre structurel : lire dans les équations si un système PEUT prévoir.

Règle du jeu : uniquement les équations et leurs paramètres, SANS entrée pilotée
et sans tâche. On a le droit de laisser le système se relâcher sans entrée (u = 0)
et de le linéariser autour de son équilibre.

Quatre conditions nécessaires à une mémoire évanescente (oui / non) :
  Q1  pas de mode neutre       : aucune direction où le système n'oublie jamais
                                 (valeur propre de la linéarisation |mu| ~ 1)
  Q2  retour au même état      : depuis des départs différents, il revient au
                                 même état en un temps raisonnable (sinon : plusieurs
                                 équilibres, ou des directions si molles qu'il y glisse)
  Q3  l'entrée agit            : l'entrée a un effet sur l'état (oui / non)
  Q4  la lecture voit          : les canaux lus dépendent de l'état (oui / non)

Q3 et Q4 sont volontairement qualitatives. Une première version demandait « au moins
2 directions effectives » : seuil arbitraire, qui rejetait à tort deux ESN excellents
(NRMSE ~0.1). Les quantités `reach` et `see` restent calculées : elles serviront à
l'analyse QUANTITATIVE, pas au filtre.

Un système admis peut prévoir ; un système rejeté ne le peut pas (hypothèse à
valider). Le filtre juge un système À UN POINT DE FONCTIONNEMENT.

Notebook : exécuter d'abord les cellules des étapes 1 à 11.
La cellule exécute les milestones Z1 à Z4, puis : val = validate_filter()
"""

import time

import numpy as np

# Notebook : pas d'import entre fichiers.

NEUTRAL_TOL = 1e-5        # |mu| > 1 - 1e-5 : direction neutre (une lente dissipation donne ~1e-3)
EQ_STARTS = 6             # départs pour compter les équilibres

_T0 = time.perf_counter()


def milestone(tag, ok, msg):
    status = "OK  " if ok else "FAIL"
    print(f"[{tag}] [{status}] {msg:<62} ({time.perf_counter() - _T0:5.2f} s)")
    return ok


def result(msg):
    print(f"[RES] {msg}")


# ------------------------------------------------------------- outils
def relax(sub, s, max_steps=8000, tol=1e-9):
    """Laisse le système évoluer sans entrée jusqu'à l'arrêt. Renvoie (état, convergé ?)."""
    for _ in range(max_steps):
        s_new = sub.step(s, 0.0)
        if np.linalg.norm(s_new - s) < tol * (1 + np.linalg.norm(s)):
            return s_new, True
        s = s_new
    return s, False


def step_jacobian(sub, s, eps=1e-6):
    """Jacobienne de l'application « un pas d'entrée » (u = 0), différences centrées."""
    n = len(s)
    J = np.empty((n, n))
    for i in range(n):
        d = np.zeros(n); d[i] = eps
        J[:, i] = (sub.step(s + d, 0.0) - sub.step(s - d, 0.0)) / (2 * eps)
    return J


def input_vector(sub, s, eps=1e-6):
    """b = d(étape)/du : comment l'entrée pousse l'état à l'équilibre."""
    return (sub.step(s, eps) - sub.step(s, -eps)) / (2 * eps)


def observe_matrix(sub, s, eps=1e-6):
    n = len(s)
    return np.stack([(sub.observe(s + eps * np.eye(n)[i]) - sub.observe(s - eps * np.eye(n)[i])) / (2 * eps)
                     for i in range(n)], axis=1)


def _eff_rank(M):
    """Rang effectif (rapport de participation des valeurs singulières au carré)."""
    sv = np.linalg.svd(M, compute_uv=False) ** 2
    return float(sv.sum() ** 2 / (sv ** 2).sum()) if sv.sum() > 0 else 0.0


def structural_report(sub, seed=0, horizon=200):
    """La fiche du filtre, calculée sans jamais piloter le système."""
    rng = np.random.default_rng(seed)
    s_eq, conv = relax(sub, sub.init_state(rng))
    J = step_jacobian(sub, s_eq)
    mu = np.abs(np.linalg.eigvals(J))
    n_neutral = int(np.sum(mu > 1 - NEUTRAL_TOL))

    # Q2 : d'autres départs mènent-ils au même équilibre ?
    starts = [sub.random_state(rng) for _ in range(EQ_STARTS)]
    ends = [relax(sub, st)[0] for st in starts]
    # Échelle = taille de la région explorée. (Correction d'un bug : l'échelle était
    # |s_eq| seule ; pour un équilibre à l'origine, le seuil devenait ~1e-12 et le
    # moindre résidu numérique comptait comme un « autre état ».)
    scale = max(np.linalg.norm(s_eq), np.mean([np.linalg.norm(st) for st in starts]))
    distinct = [s_eq]
    for e in ends:
        if all(np.linalg.norm(e - d) > 1e-3 * scale for d in distinct):
            distinct.append(e)

    # Q3 / Q4 : l'entrée atteint-elle, la lecture voit-elle ? (sur `horizon` pas)
    b, Cm = input_vector(sub, s_eq), observe_matrix(sub, s_eq)
    K, v = [], b.copy()
    for _ in range(horizon):
        K.append(v); v = J @ v
    reach = _eff_rank(np.array(K).T)
    O, w = [], Cm.copy()
    for _ in range(horizon):
        O.append(w); w = w @ J
    see = _eff_rank(np.vstack(O))

    rep = {"converged": conv, "rho": float(mu.max()), "n_neutral": n_neutral,
           "n_equilibria": len(distinct), "reach": reach, "see": see,
           "slowest": float(-1 / np.log(min(mu.max(), 1 - 1e-12)))}       # temps le plus long
    rep["Q1"], rep["Q2"] = n_neutral == 0, len(distinct) == 1
    rep["Q3"], rep["Q4"] = np.linalg.norm(b) > 1e-12, np.linalg.norm(Cm) > 1e-12
    rep["admis"] = rep["Q1"] and rep["Q2"] and rep["Q3"] and rep["Q4"]
    return rep


# ------------------------------------------------- Coulomb à symétrie brisée
class CoulombBroken(CoulombPositions):
    """Puits elliptique (k_x != k_y) et quelques charges fixes (ancrages) :
    la symétrie de rotation du puits isotrope disparaît."""

    def __init__(self, k_ratio=1.75, n_pins=0, pin_charge=1.0, **kw):
        super().__init__(**kw)
        rng = np.random.default_rng(kw.get("seed", 0) + 500)
        self.kxy = np.array([self.k_well, self.k_well * k_ratio])
        self.pins = rng.uniform(-2, 2, (n_pins, 2))
        self.qp = rng.choice([-1, 1], n_pins) * pin_charge

    def accel(self, pos, vel, u):
        a = self.coulomb_force(pos) - self.kxy * pos - self.gamma * vel
        if len(self.pins):
            d = pos[:, None, :] - self.pins[None]                       # (N, P, 2)
            inv = (np.sum(d ** 2, -1) + self.eps ** 2) ** -1.5
            a += self.k * np.sum((self.q * self.qp[None, :] * inv)[:, :, None] * d, axis=1)
        a[:, 0] += self.q[:, 0] * self.E0 * u
        return a


# ================================================================= milestones
def self_test():
    print("=" * 84)
    print("Étape 12 — validation des outils du filtre")
    print("=" * 84)
    # Z1 : la jacobienne numérique de l'ESN = la formule à la main
    sub = ESNSubstrate(rho=0.9)
    s_eq, conv = relax(sub, sub.init_state())
    e = sub.esn
    pre = e.W @ s_eq + e.b
    J_true = (1 - e.alpha) * np.eye(e.N) + e.alpha * (1 - np.tanh(pre) ** 2)[:, None] * e.W
    err = np.abs(step_jacobian(sub, s_eq) - J_true).max()
    milestone("Z1", conv and err < 1e-8, f"ESN : jacobienne numérique = analytique (écart {err:.1e})")

    # Z2 : le Coulomb isotrope SANS champ a un mode neutre, et c'est la rotation
    cou = CoulombPositions(damping=1.0, E0=0.0)
    s_c, _ = relax(cou, cou.init_state())
    w, V = np.linalg.eig(step_jacobian(cou, s_c))
    i = np.argmax(np.abs(w))
    pos = s_c[: 2 * cou.n].reshape(-1, 2)
    rot = np.concatenate([np.stack([-pos[:, 1], pos[:, 0]], 1).ravel(), np.zeros(2 * cou.n)])
    overlap = abs(np.real(V[:, i]) @ rot) / (np.linalg.norm(np.real(V[:, i])) * np.linalg.norm(rot))
    milestone("Z2", abs(abs(w[i]) - 1) < NEUTRAL_TOL and overlap > 0.99,
              f"Coulomb isotrope : |mu| = {abs(w[i]):.7f}, et c'est la rotation (recouvrement {overlap:.3f})")

    # Z3 : briser la symétrie fait disparaître ce mode neutre
    br = CoulombBroken(damping=1.0, E0=0.0)
    rb = structural_report(br)
    milestone("Z3", rb["n_neutral"] == 0, f"puits elliptique : {rb['n_neutral']} mode neutre (0 attendu), "
                                         f"|mu| max {rb['rho']:.5f}")

    # Z4 : l'ESN passe le filtre (un seul équilibre, pas de mode neutre)
    r = structural_report(ESNSubstrate(rho=0.9))
    milestone("Z4", r["admis"], f"ESN admis : {r['n_equilibria']} équilibre, {r['n_neutral']} mode neutre, "
                                f"atteinte {r['reach']:.1f}, vision {r['see']:.1f}")
    print("=" * 84)


def validate_filter(show=True):
    """Le filtre prédit-il la prévision ? Verdict calculé AVANT, performance APRÈS."""
    print("=" * 84)
    print("Validation du filtre : verdict structurel contre prévision réelle (Mackey-Glass)")
    print("=" * 84)
    s = normalize(mackey_glass())
    systems = [("ESN", g, ESNSubstrate(seed=g)) for g in range(3)]
    systems += [("Ressorts", g, MassSpringSubstrate(amp=0.3, damping=1.0, seed=g)) for g in range(12)]
    systems += [("Coulomb isotrope", g, CoulombPositions(damping=1.0, E0=0.25, seed=g)) for g in range(2)]
    systems += [("Coulomb elliptique", g, CoulombBroken(damping=1.0, E0=0.25, seed=g)) for g in range(2)]
    systems += [("Coulomb ellip.+ancrages", g, CoulombBroken(damping=1.0, E0=0.25, n_pins=6, seed=g))
                for g in range(2)]
    rows = []
    print(f"   {'système':<24}{'g':>3}  {'neutres':>7} {'équil.':>6} {'atteinte':>8} {'vision':>6}  "
          f"{'verdict':<8}{'h=10':>6}{'h=20':>6}")
    for name, g, sub in systems:
        rep = structural_report(sub, seed=g)                     # 1) le verdict, sans entrée
        nr10, nr20 = forecast_nrmse(sub, s, 10, g), forecast_nrmse(sub, s, 20, g)   # 2) la réalité
        rows.append((name, g, rep, nr10, nr20))
        print(f"   {name:<24}{g:>3}  {rep['n_neutral']:>7} {rep['n_equilibria']:>6} {rep['reach']:>8.1f} "
              f"{rep['see']:>6.1f}  {'ADMIS' if rep['admis'] else 'rejeté':<8}{nr10:6.2f}{nr20:6.2f}")
    milestone("Z5", all(np.isfinite(r[3]) for r in rows), f"{len(rows)} systèmes jugés puis testés")

    adm = np.array([r[2]["admis"] for r in rows])
    perf = np.array([(r[3] + r[4]) / 2 for r in rows])
    if adm.any() and (~adm).any():
        result(f"admis  : NRMSE moyenne {perf[adm].mean():.2f} (médiane {np.median(perf[adm]):.2f}, n={adm.sum()})")
        result(f"rejetés: NRMSE moyenne {perf[~adm].mean():.2f} (médiane {np.median(perf[~adm]):.2f}, n={(~adm).sum()})")
        worst_admitted, best_rejected = perf[adm].max(), perf[~adm].min()
        result(f"pire admis {worst_admitted:.2f}, meilleur rejeté {best_rejected:.2f} : "
               + ("séparation parfaite" if worst_admitted < best_rejected else "les deux groupes se chevauchent"))
        k = int(np.sum(np.cumsum(adm[np.argsort(perf)]) == np.arange(1, len(perf) + 1)))
        result(f"les {k} meilleurs systèmes sont tous admis (condition nécessaire, pas suffisante)")
    for q in ("Q1", "Q2", "Q3", "Q4"):
        fail = np.array([not r[2][q] for r in rows])
        if fail.any():
            result(f"{q} échoue pour {fail.sum()} systèmes : NRMSE moyenne {perf[fail].mean():.2f}")
    print("=" * 84)
    return rows


if __name__ == "__main__":
    self_test()


## Step 13 — `confirm.py`

mass chains, first confirmation of the filter

In [ ]:
"""
Étape 13 — Confirmation : le filtre FIGÉ, sur des systèmes jamais vus.

Le filtre de l'étape 12 n'est plus modifié. On l'applique à :
  - 24 nouvelles géométries de ressorts (graines 12 à 35), mêmes réglages ;
  - 3 nouveaux ESN (graines 3 à 5) ;
  - un NOUVEAU type de substrat : une chaîne de 50 masses sur une ligne,
      m x_i'' = k_c (x_{i+1} - 2 x_i + x_{i-1}) - V'(x_i) - gamma x_i' + w_i u
    en trois variantes construites pour tester chaque condition :
      a) puits simple, extrémités fixées      -> PRÉDIT ADMIS
      b) double puits (V bistable), fixées     -> PRÉDIT REJETÉ par Q2 (multistable)
      c) sans potentiel, extrémités libres     -> PRÉDIT REJETÉ par Q1 (translation neutre)

Critères de réussite, fixés AVANT l'exécution :
  K1  NRMSE médiane des admis < NRMSE médiane des rejetés
  K2  parmi le quart des systèmes qui prévoient le mieux, au moins 75 % sont admis
  K3  les trois variantes de chaîne reçoivent le verdict prédit

Notebook : exécuter d'abord les cellules des étapes 1 à 12.
La cellule exécute les milestones H1 et H2, puis : conf = confirm()   (~25 min)
"""

import time

import numpy as np

# Notebook : pas d'import entre fichiers.

_T0 = time.perf_counter()


def milestone(tag, ok, msg):
    status = "OK  " if ok else "FAIL"
    print(f"[{tag}] [{status}] {msg:<62} ({time.perf_counter() - _T0:5.2f} s)")
    return ok


def result(msg):
    print(f"[RES] {msg}")


class ChainSubstrate(Substrate):
    """Chaîne de n masses (déplacement 1D). V(x) = a x²/2 + b x⁴/4 :
    a > 0 puits simple, a < 0 double puits. fixed=False : extrémités libres."""

    def __init__(self, n=50, a=0.5, b=0.5, k_c=1.0, damping=0.5, amp=0.3, fixed=True,
                 dt=0.05, n_sub=4, seed=0):
        rng = np.random.default_rng(seed)
        self.n, self.a, self.b, self.k_c, self.gamma = n, a, b, k_c, damping
        self.fixed, self.dt, self.n_sub = fixed, dt, n_sub
        self.w_in = rng.uniform(-1, 1, n) * amp
        self.n_obs = n

    def accel(self, x, v, u):
        if self.fixed:
            xp = np.concatenate([[0.0], x, [0.0]])                       # murs à 0
        else:
            xp = np.concatenate([[x[0]], x, [x[-1]]])                    # bords libres
        lap = xp[2:] - 2 * x + xp[:-2]
        return self.k_c * lap - self.a * x - self.b * x ** 3 - self.gamma * v + self.w_in * u

    def init_state(self, rng=None):
        x = 0.01 * np.sin(np.arange(self.n))                             # pas exactement 0 :
        return np.concatenate([x, np.zeros(self.n)])                     # 0 est instable si a < 0

    def random_state(self, rng):
        return np.concatenate([rng.uniform(-1, 1, self.n), np.zeros(self.n)])

    def step(self, s, u):
        x, v = s[: self.n], s[self.n:]
        acc = self.accel(x, v, u)
        for _ in range(self.n_sub):
            v_half = v + 0.5 * acc * self.dt
            x = x + v_half * self.dt
            acc = self.accel(x, v_half, u)
            v = v_half + 0.5 * acc * self.dt
        return np.concatenate([x, v])

    def observe(self, s):
        return s[: self.n]


CHAINS = {"chaîne a) puits simple": dict(a=0.5, b=0.5, fixed=True),
          "chaîne b) double puits": dict(a=-0.5, b=1.0, fixed=True),
          "chaîne c) libre": dict(a=0.0, b=0.0, fixed=False)}
PREDICTED = {"chaîne a) puits simple": True, "chaîne b) double puits": False, "chaîne c) libre": False}


def self_test():
    print("=" * 84)
    print("Étape 13 — validation de la chaîne")
    print("=" * 84)
    probs = sum((check_substrate(ChainSubstrate(**kw)) for kw in CHAINS.values()), [])
    milestone("H1", not probs, "les trois variantes de chaîne passent le contrôle technique")
    for p in probs:
        print("   -", p)
    # H2 : la physique de base. Double puits : minima à ±sqrt(-a/b) pour une masse isolée.
    ch = ChainSubstrate(k_c=0.0, **{k: v for k, v in CHAINS["chaîne b) double puits"].items()})
    x_eq, _ = relax(ch, np.concatenate([np.full(ch.n, 0.3), np.zeros(ch.n)]))
    target = np.sqrt(0.5 / 1.0)
    milestone("H2", np.allclose(x_eq[: ch.n], target, atol=1e-4),
              f"double puits sans couplage : équilibre à {x_eq[0]:.4f} (théorie {target:.4f})")
    print("=" * 84)


def confirm():
    print("=" * 84)
    print("Confirmation du filtre figé sur des systèmes jamais vus")
    print("=" * 84)
    print("   Prédictions écrites avant l'exécution :")
    for name, adm in PREDICTED.items():
        print(f"      {name:<26} -> {'ADMIS' if adm else 'REJETÉ'}")
    s = normalize(mackey_glass())
    systems = [("ESN", g, ESNSubstrate(seed=g)) for g in range(3, 6)]
    systems += [("Ressorts", g, MassSpringSubstrate(amp=0.3, damping=1.0, seed=g)) for g in range(12, 36)]
    systems += [(name, g, ChainSubstrate(seed=g, **kw)) for name, kw in CHAINS.items() for g in range(2)]
    rows = []
    print(f"\n   {'système':<26}{'g':>3}  {'neutres':>7} {'états':>6}  {'verdict':<8}{'h=10':>6}{'h=20':>6}")
    for name, g, sub in systems:
        rep = structural_report(sub, seed=g)
        nr = (forecast_nrmse(sub, s, 10, g), forecast_nrmse(sub, s, 20, g))
        rows.append((name, g, rep, np.mean(nr)))
        print(f"   {name:<26}{g:>3}  {rep['n_neutral']:>7} {rep['n_equilibria']:>6}  "
              f"{'ADMIS' if rep['admis'] else 'rejeté':<8}{nr[0]:6.2f}{nr[1]:6.2f}")
    milestone("H3", all(np.isfinite(r[3]) for r in rows), f"{len(rows)} systèmes jugés puis testés")

    adm = np.array([r[2]["admis"] for r in rows])
    perf = np.array([r[3] for r in rows])
    print()
    k1 = adm.any() and (~adm).any() and np.median(perf[adm]) < np.median(perf[~adm])
    result(f"K1 : médiane admis {np.median(perf[adm]):.2f} contre rejetés {np.median(perf[~adm]):.2f} "
           f"-> {'RÉUSSI' if k1 else 'ÉCHOUÉ'}")
    top = np.argsort(perf)[: max(1, len(perf) // 4)]
    frac = adm[top].mean()
    result(f"K2 : {adm[top].sum()}/{len(top)} des meilleurs sont admis ({frac:.0%}) "
           f"-> {'RÉUSSI' if frac >= 0.75 else 'ÉCHOUÉ'}")
    k3 = all(r[2]["admis"] == PREDICTED[r[0]] for r in rows if r[0] in PREDICTED)
    result(f"K3 : verdicts des chaînes conformes aux prédictions -> {'RÉUSSI' if k3 else 'ÉCHOUÉ'}")
    for name in PREDICTED:
        sel = [r for r in rows if r[0] == name]
        why = ", ".join(q for q in ("Q1", "Q2", "Q3", "Q4") if not sel[0][2][q]) or "aucune condition échouée"
        result(f"    {name:<26} {'ADMIS' if sel[0][2]['admis'] else 'rejeté'} ({why}), "
               f"NRMSE {np.mean([r[3] for r in sel]):.2f}")
    ms = np.array([r[0] == "Ressorts" for r in rows])
    if (adm & ms).any() and (~adm & ms).any():
        result(f"nouvelles géométries : {np.sum(adm & ms)} admises (médiane {np.median(perf[adm & ms]):.2f}), "
               f"{np.sum(~adm & ms)} rejetées (médiane {np.median(perf[~adm & ms]):.2f})")
    print("=" * 84)
    return rows


if __name__ == "__main__":
    self_test()


## Step 14 — `linear_mc.py`

memory capacity from the linearisation

In [ ]:
"""
Étape 14 — La capacité mémoire prédite par les équations, sans paramètre ajustable.

Autour de son équilibre, un système piloté se comporte comme sa linéarisation :
    s(t+1) = J s(t) + b u(t),      x(t) = C s(t)   (canaux lus)
donc x(t) = Σ_j g_j u(t-j) avec g_j = C J^j b. Pour une entrée i.i.d., la part de
u(t-k) qu'une lecture linéaire retrouve dans les canaux bruités vaut
    MC_k = g_k^T (G + eps² diag G)^{-1} g_k,      G = Σ_j g_j g_j^T,
où eps est le bruit de mesure relatif (1 %). La capacité est MC = Σ_{k=1..80} MC_k.
Tout vient de J, b, C : aucune simulation pilotée, aucun paramètre libre.

Ce fichier définit aussi la NON-LINÉARITÉ STATIQUE : on applique une entrée
constante u = 1 puis u = 0.5 et on compare les déplacements d'équilibre,
    nl = |dx(1) - 2 dx(0.5)| / |dx(1)|      (0 pour un système linéaire).

Critère fixé AVANT l'exécution de linear_mc_test() :
  P1  pour au moins 80 % des systèmes presque linéaires (nl < 0.02), la capacité
      mesurée est à moins de 20 % de la capacité prédite.

Notebook : exécuter d'abord les cellules des étapes 1 à 13.
La cellule exécute les milestones T1 à T4, puis : lin = linear_mc_test()   (~5 min)
"""

import time

import numpy as np

# Notebook : pas d'import entre fichiers.

N_LONG = 20000            # pas de mesure pour la comparaison à la théorie (voir measure_mc)

_T0 = time.perf_counter()


def milestone(tag, ok, msg):
    status = "OK  " if ok else "FAIL"
    print(f"[{tag}] [{status}] {msg:<62} ({time.perf_counter() - _T0:5.2f} s)")
    return ok


def result(msg):
    print(f"[RES] {msg}")


# ------------------------------------------------------------- la théorie
def linear_memory_capacity(J, b, Cm, k_max=K_MAX, noise=NOISE, horizon=3000):
    """Capacité mémoire du système linéarisé. Renvoie (MC, profil MC_k pour k = 1..k_max)."""
    if np.max(np.abs(np.linalg.eigvals(J))) > 1 + 1e-6:     # équilibre instable : la
        return float("nan"), np.full(k_max, np.nan)          # linéarisation n'a pas de sens
    g, v = [], b.copy()
    for _ in range(horizon):                      # g_j = C J^j b
        g.append(Cm @ v)
        v = J @ v
    g = np.array(g)
    G = g.T @ g                                   # covariance des canaux (entrée de variance 1)
    R = G + noise ** 2 * np.diag(np.diag(G)) + 1e-300 * np.eye(len(G))
    live = np.diag(G) > 1e-24 * np.diag(G).max()  # canaux qui ne reçoivent rien : retirés
    Rinv = np.linalg.pinv(R[np.ix_(live, live)], hermitian=True)
    gk = g[1: k_max + 1][:, live]
    mck = np.einsum("ki,ij,kj->k", gk, Rinv, gk)
    return float(mck.sum()), mck


def linearize(sub, s_eq=None):
    """(s_eq, J, b, C) autour de l'équilibre sans entrée."""
    if s_eq is None:
        s_eq, _ = relax(sub, sub.init_state())
    return s_eq, step_jacobian(sub, s_eq), input_vector(sub, s_eq), observe_matrix(sub, s_eq)


# ------------------------------------------------------------- non-linéarité statique
def static_response(sub, s0, u, max_steps=8000, tol=1e-9):
    """État atteint sous une entrée CONSTANTE u, en partant de s0."""
    s = s0
    for _ in range(max_steps):
        s_new = sub.step(s, u)
        if np.linalg.norm(s_new - s) < tol * (1 + np.linalg.norm(s)):
            return s_new
        s = s_new
    return s


def static_nonlinearity(sub, s_eq, u=1.0):
    """Écart à la proportionnalité de la réponse statique, vu par les canaux lus."""
    x0 = sub.observe(s_eq)
    d1 = sub.observe(static_response(sub, s_eq, u)) - x0
    dh = sub.observe(static_response(sub, s_eq, u / 2)) - x0
    n1 = np.linalg.norm(d1)
    return float(np.linalg.norm(d1 - 2 * dh) / n1) if n1 > 1e-12 else 0.0


# ------------------------------------------------------------- la mesure
def measure_mc(sub, seed=0, n=None):
    """Capacité mémoire mesurée (banc honnête : lecture testée sur un autre départ, autre entrée).

    n : nombre de pas. La formule théorique donne la limite d'une lecture apprise sur
    une infinité de données. Avec n = 3000 (banc habituel) et 50 canaux, la lecture est
    un peu sur-ajustée et la mesure lit ~10 % trop bas (ESN linéaire : 8.95 pour 10.05 ;
    9.70 à n = 10 000 ; 9.93 à n = 100 000). On compare donc à la théorie avec N_LONG pas.
    """
    n = T if n is None else n
    u = np.random.default_rng(42).uniform(-1, 1, WARMUP + n)
    u_test = np.random.default_rng(7).uniform(-1, 1, WARMUP + n)
    X, _ = run(sub, u)
    Xd, _ = run(sub, u_test, sub.random_state(np.random.default_rng(1000 + seed)))
    return memory_capacity(X[WARMUP:], u[WARMUP:], Xd[WARMUP:], u_test[WARMUP:])[0]


class LinearESN(ESNSubstrate):
    """ESN dont on a retiré le tanh : exactement linéaire, pour tester la formule."""

    def step(self, s, u):
        e = self.esn
        return (1 - e.alpha) * s + e.alpha * (e.W @ s + e.W_in * u + e.b)


class _Scalar(Substrate):
    """x(t+1) = a x(t) + u(t) : un seul canal, capacité connue exactement."""
    n_obs = 1

    def __init__(self, a=0.5):
        self.a = a

    def init_state(self, rng=None):
        return np.zeros(1)

    def step(self, s, u):
        return self.a * s + u

    def observe(self, s):
        return s


# ================================================================= milestones
def self_test():
    print("=" * 84)
    print("Étape 14 — validation de la capacité mémoire théorique")
    print("=" * 84)
    # T1 : un canal, x(t+1) = a x + u. r²_k = a^{2k} (1 - a²) / (1 + eps²), somme = a² / (1 + eps²)
    sc = _Scalar(0.5)
    _, J, b, Cm = linearize(sc)
    mc, mck = linear_memory_capacity(J, b, Cm)
    exact = 0.25 / (1 + NOISE ** 2)
    milestone("T1", abs(mc - exact) < 1e-9 and abs(mck[0] - 0.1875 / (1 + NOISE ** 2)) < 1e-9,
              f"système scalaire : MC = {mc:.6f} (exact {exact:.6f})")

    # T2 : ESN exactement linéaire, la formule doit donner la capacité mesurée
    lin = LinearESN(rho=0.9, alpha=0.3)
    _, J, b, Cm = linearize(lin)
    mc_th, mc_me = linear_memory_capacity(J, b, Cm)[0], measure_mc(lin, n=30000)
    mc_short = measure_mc(lin)
    milestone("T2", abs(mc_me - mc_th) / mc_th < 0.05 and mc_short < mc_me,
              f"ESN linéaire : théorie {mc_th:.2f}, mesure {mc_me:.2f} (30 000 pas), {mc_short:.2f} (3000)")

    # T3 : la non-linéarité statique vaut 0 pour un système linéaire, > 0 sinon
    s_l, _ = relax(lin, lin.init_state())
    esn = ESNSubstrate(rho=0.9, alpha=0.3)
    s_e, _ = relax(esn, esn.init_state())
    nl_l, nl_e = static_nonlinearity(lin, s_l), static_nonlinearity(esn, s_e)
    milestone("T3", nl_l < 1e-6 and nl_e > 0.01,
              f"non-linéarité statique : ESN linéaire {nl_l:.1e}, ESN tanh {nl_e:.3f}")

    # T4 : ESN tanh à très faible entrée : presque linéaire, la théorie doit tenir
    small = ESNSubstrate(rho=0.9, alpha=0.3, input_scale=0.01)
    s_s, J, b, Cm = linearize(small)
    mc_th, mc_me = linear_memory_capacity(J, b, Cm)[0], measure_mc(small, n=30000)
    milestone("T4", abs(mc_me - mc_th) / mc_th < 0.10,
              f"ESN tanh, entrée x0.01 : théorie {mc_th:.2f}, mesure {mc_me:.2f} "
              f"(nl = {static_nonlinearity(small, s_s):.4f})")
    print("=" * 84)


def linear_mc_test(show=True):
    """Théorie contre mesure, sur trois familles et plusieurs amplitudes d'entrée."""
    print("=" * 84)
    print("La capacité mémoire prédite par la linéarisation tient-elle ? (critère P1)")
    print("=" * 84)
    systems = []
    for g in range(3):
        for sc in (0.01, 0.1, 0.3, 1.0, 3.0):
            systems.append(("ESN", g, sc, ESNSubstrate(seed=g, input_scale=sc)))
    for g in range(2):
        for amp in (0.03, 0.3, 1.0, 3.0):
            systems.append(("Chaîne (a)", g, amp, ChainSubstrate(seed=g, amp=amp, a=0.5, b=0.5)))
    for g in (0, 3, 7):                               # géométries admises par le filtre
        for amp in (0.03, 0.3):
            systems.append(("Ressorts", g, amp, MassSpringSubstrate(seed=g, amp=amp, damping=1.0)))
    rows = []
    print(f"   {'système':<12}{'g':>3}{'ampl.':>7}{'nl':>8}{'MC théorie':>12}{'MC mesure':>11}{'écart':>8}")
    for name, g, amp, sub in systems:
        s_eq, J, b, Cm = linearize(sub)
        mc_th = linear_memory_capacity(J, b, Cm)[0]
        nl = static_nonlinearity(sub, s_eq)
        mc_me = measure_mc(sub, g, n=N_LONG)
        rows.append((name, g, amp, nl, mc_th, mc_me))
        print(f"   {name:<12}{g:>3}{amp:>7}{nl:>8.4f}{mc_th:>12.2f}{mc_me:>11.2f}"
              f"{(mc_me - mc_th) / mc_th:>+8.0%}")
    milestone("T5", all(np.isfinite(r[4]) and np.isfinite(r[5]) for r in rows),
              f"{len(rows)} systèmes : théorie et mesure calculées")

    nl = np.array([r[3] for r in rows]); th = np.array([r[4] for r in rows]); me = np.array([r[5] for r in rows])
    err = np.abs(me - th) / th
    near = nl < 0.02
    frac = np.mean(err[near] < 0.20) if near.any() else np.nan
    result(f"P1 : {np.sum(err[near] < 0.20)}/{near.sum()} systèmes presque linéaires à moins de 20 % "
           f"de la théorie ({frac:.0%}) -> {'RÉUSSI' if frac >= 0.8 else 'ÉCHOUÉ'}")
    result(f"écart médian : {np.median(err[near]):.0%} si nl < 0.02, "
           f"{np.median(err[~near]):.0%} si nl >= 0.02")
    result(f"sens de l'écart quand le système devient non linéaire : la mesure est "
           f"{'inférieure' if np.median((me - th)[~near]) < 0 else 'supérieure'} à la théorie "
           f"dans {np.mean((me - th)[~near] < 0):.0%} des cas")

    import matplotlib.pyplot as plt
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
    marks = {"ESN": "o", "Chaîne (a)": "s", "Ressorts": "^"}
    for fam, mk in marks.items():
        m = np.array([r[0] == fam for r in rows])
        axs[0].scatter(th[m], me[m], c=np.log10(nl[m] + 1e-4), cmap="viridis", vmin=-4, vmax=0, marker=mk,
                       s=40, edgecolors="k", linewidths=0.3, label=fam)
        sc = axs[1].scatter(nl[m] + 1e-4, me[m] / th[m], c=np.log10(nl[m] + 1e-4), cmap="viridis", vmin=-4,
                            vmax=0, marker=mk, s=40, edgecolors="k", linewidths=0.3, label=fam)
    lim = max(th.max(), me.max()) * 1.05
    axs[0].plot([0, lim], [0, lim], "k--", lw=1)
    axs[0].set(xlabel="capacité mémoire prédite (équations seules)", ylabel="capacité mémoire mesurée",
               title="Théorie contre mesure")
    axs[0].legend(fontsize=8)
    axs[1].axhline(1, color="k", ls="--", lw=1); axs[1].axvline(0.02, color="0.6", lw=0.8)
    axs[1].set(xscale="log", xlabel="non-linéarité statique nl", ylabel="mesure / théorie",
               title="L'accord se dégrade quand nl augmente")
    fig.colorbar(sc, ax=axs[1], label="log10(nl)")
    fig.tight_layout()
    fig.savefig("linear_mc.png", dpi=110)
    milestone("T6", True, "figure enregistrée : linear_mc.png")
    print("=" * 84)
    if show:
        plt.show()
    return rows


if __name__ == "__main__":
    self_test()


## Step 15 — `features.py`

graded structural profile, two new substrates

In [ ]:
"""
Étape 15 — Le profil structurel gradué, et deux nouvelles familles de substrats.

Le filtre de l'étape 12 répond par oui ou non. Le profil ajoute des DEGRÉS, toujours
calculés sans piloter le système par une série (seules sont autorisées : l'évolution
sans entrée, une perturbation infinitésimale, une entrée constante) :

  mc_lin   capacité mémoire prédite par la linéarisation (étape 14)
  nl       non-linéarité statique à l'amplitude de travail (étape 14)
  d2       version graduée de Q2 : distance moyenne entre les états finaux obtenus
           depuis 6 départs et l'équilibre de référence, rapportée à l'amplitude du
           déplacement statique sous entrée, |x(u=+1) - x(u=-1)| / 2.
           d2 = 0 : le système revient toujours au même état ; d2 << 1 : les états
           finaux diffèrent moins que ce que l'entrée fait bouger ; d2 >> 1 : ils
           diffèrent plus.
  + les grandeurs du filtre : modes neutres, nombre d'états finaux, temps le plus lent.

Nouveaux substrats :
  DelayNodeSubstrate   un seul nœud non linéaire et une boucle de retard, 50 nœuds
                       virtuels le long du retard (le « graphe dans le temps »)
  CoulombRepulsive     charges toutes de même signe dans un puits elliptique

Notebook : exécuter d'abord les cellules des étapes 1 à 14.
La cellule exécute les milestones U1 à U5.
"""

import time

import numpy as np

# Notebook : pas d'import entre fichiers.

_T0 = time.perf_counter()


def milestone(tag, ok, msg):
    status = "OK  " if ok else "FAIL"
    print(f"[{tag}] [{status}] {msg:<62} ({time.perf_counter() - _T0:5.2f} s)")
    return ok


# ------------------------------------------------------------- nouveaux substrats
class DelayNodeSubstrate(Substrate):
    """Nœud unique à retard (idée d'Appeltant et al. 2011), version à temps discret.
    Les n nœuds virtuels sont parcourus dans l'ordre le long de la boucle :
        s_i(t) = d s_{i-1}(t) + (1 - d) tanh(eta s_i(t-1) + gamma m_i u(t) + phi),
    avec s_0(t) = s_n(t-1), d = exp(-theta), et m_i = ±1 un masque fixe."""

    def __init__(self, n=50, eta=0.8, gamma=0.5, theta=0.5, phi=0.2, seed=0):
        rng = np.random.default_rng(seed)
        self.n, self.eta, self.gamma, self.d, self.phi = n, eta, gamma, np.exp(-theta), phi
        self.mask = rng.choice([-1.0, 1.0], n)
        self.n_obs = n

    def init_state(self, rng=None):
        return np.zeros(self.n)

    def random_state(self, rng):
        return rng.uniform(-1, 1, self.n)

    def step(self, s, u):
        drive = (1 - self.d) * np.tanh(self.eta * s + self.gamma * self.mask * u + self.phi)
        new, prev = np.empty(self.n), s[-1]
        for i in range(self.n):                       # inertie le long de la boucle
            prev = self.d * prev + drive[i]
            new[i] = prev
        return new

    def observe(self, s):
        return s


class CoulombRepulsive(CoulombBroken):
    """Charges toutes positives (0.3 à 1) dans un puits elliptique : plus d'attraction
    entre charges opposées, donc a priori moins d'arrangements d'équilibre."""

    def __init__(self, **kw):
        super().__init__(**kw)
        rng = np.random.default_rng(kw.get("seed", 0) + 900)
        self.q = rng.uniform(0.3, 1.0, (self.n, 1))
        self.qq = self.q * self.q.T
        np.fill_diagonal(self.qq, 0.0)


# ------------------------------------------------------------- le profil
def structural_profile(sub, seed=0, horizon=200):
    """Le filtre de l'étape 12 (mêmes tirages, donc mêmes verdicts) et les degrés."""
    rng = np.random.default_rng(seed)
    s_eq, conv = relax(sub, sub.init_state(rng))
    J = step_jacobian(sub, s_eq)
    mu = np.abs(np.linalg.eigvals(J))
    n_neutral = int(np.sum(mu > 1 - NEUTRAL_TOL))
    starts = [sub.random_state(rng) for _ in range(EQ_STARTS)]
    ends = [relax(sub, st)[0] for st in starts]
    scale = max(np.linalg.norm(s_eq), np.mean([np.linalg.norm(st) for st in starts]))
    distinct = [s_eq]
    for e in ends:
        if all(np.linalg.norm(e - d) > 1e-3 * scale for d in distinct):
            distinct.append(e)
    b, Cm = input_vector(sub, s_eq), observe_matrix(sub, s_eq)

    # Degrés : réponses statiques à une entrée constante
    x0 = sub.observe(s_eq)
    xp = sub.observe(static_response(sub, s_eq, 1.0))
    xh = sub.observe(static_response(sub, s_eq, 0.5))
    xm = sub.observe(static_response(sub, s_eq, -1.0))
    n1 = np.linalg.norm(xp - x0)
    nl = float(np.linalg.norm((xp - x0) - 2 * (xh - x0)) / n1) if n1 > 1e-12 else 0.0
    amp = 0.5 * np.linalg.norm(xp - xm)
    spread = float(np.mean([np.linalg.norm(sub.observe(e) - x0) for e in ends]))
    d2 = spread / amp if amp > 1e-12 else np.inf

    p = {"converged": conv, "rho": float(mu.max()), "n_neutral": n_neutral,
         "n_equilibria": len(distinct),
         "slowest": float(-1 / np.log(min(mu.max(), 1 - 1e-12))),
         "mc_lin": linear_memory_capacity(J, b, Cm)[0], "nl": nl, "d2": float(d2),
         "static_amp": float(amp), "spread": spread}
    p["Q1"], p["Q2"] = n_neutral == 0, len(distinct) == 1
    p["Q3"], p["Q4"] = bool(np.linalg.norm(b) > 1e-12), bool(np.linalg.norm(Cm) > 1e-12)
    p["admis"] = bool(p["Q1"] and p["Q2"] and p["Q3"] and p["Q4"])
    return p


# ================================================================= milestones
def self_test():
    print("=" * 84)
    print("Étape 15 — validation du profil structurel et des nouveaux substrats")
    print("=" * 84)
    new = [DelayNodeSubstrate(), DelayNodeSubstrate(eta=1.5, phi=0.0), CoulombRepulsive(damping=1.0, E0=0.25)]
    probs = sum((check_substrate(s) for s in new), [])
    milestone("U1", not probs, "nœud à retard et Coulomb répulsif : contrôle technique")
    for pb in probs:
        print("   -", pb)

    # U2 : le profil reproduit exactement les verdicts du filtre de l'étape 12
    cases = [("ESN", ESNSubstrate(seed=0))] + [(k, ChainSubstrate(seed=0, **kw)) for k, kw in CHAINS.items()]
    profs = {name: structural_profile(sub) for name, sub in cases}
    same = all(profs[name]["admis"] == structural_report(sub)["admis"]
               and profs[name]["n_equilibria"] == structural_report(sub)["n_equilibria"]
               for name, sub in cases)
    milestone("U2", same, "profil et filtre de l'étape 12 : mêmes verdicts sur 4 systèmes")

    # U3 : les degrés sur des cas connus
    lin = structural_profile(LinearESN(rho=0.9, alpha=0.3))
    d2_esn, d2_dw = profs["ESN"]["d2"], profs["chaîne b) double puits"]["d2"]
    milestone("U3", lin["nl"] < 1e-6 and d2_esn < 1e-6 and d2_dw > 1,
              f"nl (ESN linéaire) = {lin['nl']:.0e} ; d2 : ESN {d2_esn:.0e}, double puits {d2_dw:.1f}")

    # U4 : le nœud à retard avec eta > 1 est bistable, avec eta < 1 il ne l'est pas
    mono, bi = structural_profile(new[0]), structural_profile(new[1])
    milestone("U4", mono["admis"] and bi["n_equilibria"] >= 2,
              f"nœud à retard : eta=0.8 -> {mono['n_equilibria']} état (admis), "
              f"eta=1.5 -> {bi['n_equilibria']} états")

    # U5 : la capacité linéaire du profil est celle de l'étape 14
    esn = ESNSubstrate(seed=0)
    _, J, b, Cm = linearize(esn)
    milestone("U5", abs(profs["ESN"]["mc_lin"] - linear_memory_capacity(J, b, Cm)[0]) < 1e-9,
              f"mc_lin du profil = étape 14 ({profs['ESN']['mc_lin']:.2f})")
    for name, pr in list(profs.items()) + [("nœud à retard", mono), ("Coulomb répulsif", structural_profile(new[2]))]:
        print(f"   {name:<24} neutres {pr['n_neutral']}  états {pr['n_equilibria']}  mc_lin {pr['mc_lin']:6.2f}  "
              f"nl {pr['nl']:.3f}  d2 {pr['d2']:8.3f}  {'ADMIS' if pr['admis'] else 'rejeté'}")
    print("=" * 84)


if __name__ == "__main__":
    self_test()


## Step 16 — `explore.py`

exploration on already-seen systems

In [ ]:
"""
Étape 16 — Exploration : que disent les degrés structurels sur les systèmes DÉJÀ VUS ?

Pour chaque système : le profil structurel (étape 15), calculé d'abord et sans série,
puis l'erreur de prévision de Mackey-Glass à 10 et 20 pas (étape 11).

L'ensemble d'exploration ne contient que des systèmes dont la performance avait déjà
été regardée (validation et confirmation du filtre, essai sur l'amplitude de la chaîne).
On a donc le droit d'y chercher librement ; ce qu'on y trouve est ensuite GELÉ, puis
testé à l'étape 17 sur des systèmes jamais vus.

Deux choses sont choisies ici :
  1. le seuil theta de la règle graduée « admis si aucun mode neutre et d2 < theta » ;
  2. un prédicteur de l'erreur à partir du profil, parmi deux candidats fixés d'avance
     (régression ridge ou 5 plus proches voisins), jugés en retirant un système à la fois.

Notebook : exécuter d'abord les cellules des étapes 1 à 15.
Puis : rows = run_catalog(EXPLORATION, "exploration.jsonl")     (~45 min)
       frozen = explore_analysis(rows)
"""

import json
import os
import time

import numpy as np

# Notebook : pas d'import entre fichiers.

_T0 = time.perf_counter()
GOOD = 0.25                       # « bonne prévision » : NRMSE moyenne < 0.25
FEATURES = ["log_mc", "log_nl", "log_nl2", "log_d2", "neutral"]


def milestone(tag, ok, msg):
    status = "OK  " if ok else "FAIL"
    print(f"[{tag}] [{status}] {msg:<62} ({time.perf_counter() - _T0:5.2f} s)")
    return ok


def result(msg):
    print(f"[RES] {msg}")


# ------------------------------------------------------------- le catalogue
def make_system(family, seed, **kw):
    if family == "ESN":
        return ESNSubstrate(seed=seed, **kw)
    if family == "Ressorts":
        return MassSpringSubstrate(seed=seed, **kw)
    if family == "Coulomb":
        kw = dict(kw)
        kind = kw.pop("kind")
        if kind == "isotrope":
            return CoulombPositions(seed=seed, **kw)
        return CoulombBroken(seed=seed, n_pins=6 if kind == "ancrages" else 0, **kw)
    if family == "Chaîne":
        kw = dict(kw)
        return ChainSubstrate(seed=seed, **CHAINS[kw.pop("kind")], **kw)
    if family == "Retard":
        return DelayNodeSubstrate(seed=seed, **kw)
    if family == "Coulomb répulsif":
        return CoulombRepulsive(seed=seed, **kw)
    raise ValueError(family)


_CH = {"a": "chaîne a) puits simple", "b": "chaîne b) double puits", "c": "chaîne c) libre"}

EXPLORATION = (
    [("ESN", g, {}) for g in range(6)]
    + [("Ressorts", g, {"amp": 0.3, "damping": 1.0}) for g in range(36)]
    + [("Coulomb", g, {"kind": k, "damping": 1.0, "E0": 0.25})
       for k in ("isotrope", "elliptique", "ancrages") for g in range(2)]
    + [("Chaîne", g, {"kind": _CH[k], "amp": 0.3}) for k in "abc" for g in range(2)]
    + [("Chaîne", g, {"kind": _CH["a"], "amp": a}) for a in (0.03, 0.1, 1.0, 3.0, 10.0) for g in range(2)]
)


def evaluate_system(family, seed, kw, series):
    """Profil d'abord (sans série), prévision ensuite."""
    sub = make_system(family, seed, **kw)
    prof = structural_profile(sub, seed=seed)
    n10, n20 = forecast_nrmse(sub, series, 10, seed), forecast_nrmse(sub, series, 20, seed)
    return {"family": family, "seed": seed, "params": kw, **prof,
            "nrmse10": n10, "nrmse20": n20, "perf": (n10 + n20) / 2}


def run_catalog(specs, path, part=None):
    """Évalue les systèmes de `specs` et ajoute une ligne JSON par système à `path`.
    Reprend là où il s'était arrêté. part=(i, n) : ne traite que les systèmes d'indice
    congru à i modulo n (pour lancer n processus en parallèle, un fichier chacun)."""
    done = set()
    if os.path.exists(path):
        done = {json.loads(l)["key"] for l in open(path)}
    series = normalize(mackey_glass())
    for idx, (family, seed, kw) in enumerate(specs):
        key = f"{family}|{seed}|{json.dumps(kw, sort_keys=True, ensure_ascii=False)}"
        if key in done or (part is not None and idx % part[1] != part[0]):
            continue
        row = evaluate_system(family, seed, kw, series)
        row["key"] = key
        with open(path, "a") as f:
            f.write(json.dumps(row, ensure_ascii=False, default=float) + "\n")
        print(f"   {family:<17}{seed:>3} {str(kw)[:44]:<44} neutres {row['n_neutral']} états {row['n_equilibria']} "
              f"nl {row['nl']:.3f} d2 {row['d2']:9.3f} -> {row['nrmse10']:.2f} {row['nrmse20']:.2f}", flush=True)
    return load_rows(path)


def load_rows(*paths):
    rows = [json.loads(l) for p in paths if os.path.exists(p) for l in open(p)]
    return sorted(rows, key=lambda r: r["key"])


# ------------------------------------------------------------- descripteurs et modèles
def feature_matrix(rows):
    mc = np.array([r["mc_lin"] if r["mc_lin"] is not None and np.isfinite(r["mc_lin"]) else 0.0 for r in rows])
    nl = np.array([r["nl"] for r in rows])
    d2 = np.array([min(r["d2"], 1e3) if np.isfinite(r["d2"]) else 1e3 for r in rows])
    ln = np.log10(nl + 1e-3)
    return np.column_stack([np.log10(mc + 0.1), ln, ln ** 2, np.log10(d2 + 1e-3),
                            [float(r["n_neutral"] > 0) for r in rows]])


def _zs(F_tr, F_te):
    mu, sd = F_tr.mean(0), F_tr.std(0)
    sd = np.where(sd < 1e-9, 1.0, sd)
    return (F_tr - mu) / sd, (F_te - mu) / sd


def predict_ridge(F_tr, y_tr, F_te, alpha=1.0):
    A, B = _zs(F_tr, F_te)
    A1, B1 = np.hstack([A, np.ones((len(A), 1))]), np.hstack([B, np.ones((len(B), 1))])
    reg = alpha * np.eye(A1.shape[1]); reg[-1, -1] = 0
    return np.clip(B1 @ np.linalg.solve(A1.T @ A1 + reg, A1.T @ y_tr), 0, 1)


def predict_knn(F_tr, y_tr, F_te, k=5):
    A, B = _zs(F_tr, F_te)
    d = np.linalg.norm(B[:, None] - A[None], axis=2)
    return y_tr[np.argsort(d, axis=1)[:, :k]].mean(1)


MODELS = {"ridge": predict_ridge, "5 plus proches voisins": predict_knn}


def r2_score(y, p):
    return float(1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2))


def graded_admit(rows, theta):
    return np.array([r["n_neutral"] == 0 and r["d2"] < theta for r in rows])


def separation(adm, perf):
    """(sensibilité + spécificité - 1) pour reconnaître les bonnes prévisions (perf < GOOD)."""
    good = perf < GOOD
    if good.sum() == 0 or (~good).sum() == 0:
        return np.nan
    return float(adm[good].mean() + (~adm)[~good].mean() - 1)


# ================================================================= l'exploration
def explore_analysis(rows, show=True):
    print("=" * 84)
    print("Étape 16 — exploration sur les systèmes déjà vus")
    print("=" * 84)
    perf = np.array([r["perf"] for r in rows]); fam = np.array([r["family"] for r in rows])
    binary = np.array([r["admis"] for r in rows])
    F = feature_matrix(rows)
    milestone("E1", len(rows) == len(EXPLORATION) and np.all(np.isfinite(F)) and np.all(np.isfinite(perf)),
              f"{len(rows)} systèmes, profils et prévisions complets")

    result("par famille : effectif, admis par le filtre binaire, NRMSE médiane")
    for f in sorted(set(fam)):
        m = fam == f
        print(f"      {f:<10} n = {m.sum():>2}   admis {binary[m].sum():>2}   médiane {np.median(perf[m]):.2f}")

    result("corrélation de rang entre chaque degré et l'erreur (+ : l'erreur augmente avec le degré)")
    names = ["mc_lin", "nl", "d2", "slowest"]
    cols = {n: np.array([r[n] if r[n] is not None and np.isfinite(r[n]) else np.nan for r in rows]) for n in names}
    groups = {"tous": np.ones(len(rows), bool), "admis (binaire)": binary, "ressorts": fam == "Ressorts"}
    print("      " + " " * 18 + "".join(f"{n:>10}" for n in names))
    for gname, m in groups.items():
        print(f"      {gname:<18}" + "".join(f"{spearman(cols[n][m], perf[m]):>+10.2f}" for n in names))

    # 1. Seuil de la règle graduée
    result("règle graduée « aucun mode neutre et d2 < theta » : séparation des bonnes prévisions")
    thetas = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]
    seps = [separation(graded_admit(rows, th), perf) for th in thetas]
    for th, sp in zip(thetas, seps):
        adm = graded_admit(rows, th)
        print(f"      theta = {th:<5} admis {adm.sum():>2}   séparation {sp:+.2f}   "
              f"médiane admis {np.median(perf[adm]) if adm.any() else np.nan:.2f} / "
              f"rejetés {np.median(perf[~adm]) if (~adm).any() else np.nan:.2f}")
    print(f"      filtre binaire      admis {binary.sum():>2}   séparation {separation(binary, perf):+.2f}")
    theta = thetas[int(np.nanargmax(seps))]
    result(f"seuil retenu : theta = {theta}")

    # 2. Prédicteur : deux candidats, un système retiré à la fois
    result("prédire l'erreur à partir du profil (R², un système retiré à la fois)")
    loo = {}
    for name, fn in MODELS.items():
        p = np.array([fn(np.delete(F, i, 0), np.delete(perf, i), F[i:i + 1])[0] for i in range(len(rows))])
        loo[name] = p
        print(f"      {name:<24} tous {r2_score(perf, p):5.2f}   ressorts seuls "
              f"{r2_score(perf[fam == 'Ressorts'], p[fam == 'Ressorts']):5.2f}")
    base = np.array([np.mean(np.delete(perf, i)[np.delete(fam, i) == fam[i]]) for i in range(len(rows))])
    print(f"      {'moyenne de la famille':<24} tous {r2_score(perf, base):5.2f}   ressorts seuls "
          f"{r2_score(perf[fam == 'Ressorts'], base[fam == 'Ressorts']):5.2f}")
    model = max(loo, key=lambda n: r2_score(perf, loo[n]))
    result(f"modèle retenu : {model}")
    milestone("E2", True, "seuil et modèle choisis sur l'exploration seule")

    import matplotlib.pyplot as plt
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.3))
    marks = {"ESN": "o", "Ressorts": "^", "Coulomb": "X", "Chaîne": "s"}
    for f, mk in marks.items():
        m = fam == f
        axs[0].scatter(np.clip(cols["d2"][m], 1e-9, 1e3) + 1e-9, perf[m], marker=mk, s=30, label=f)
        axs[1].scatter(cols["nl"][m] + 1e-4, perf[m], marker=mk, s=30, label=f)
        axs[2].scatter(perf[m], loo[model][m], marker=mk, s=30, label=f)
    axs[0].axvline(theta, color="tab:red", lw=1)
    axs[0].set(xscale="log", xlabel="d2 (états finaux / amplitude statique)", ylabel="NRMSE de prévision",
               title=f"Q2 graduée (seuil theta = {theta})")
    axs[1].set(xscale="log", xlabel="non-linéarité statique nl", ylabel="NRMSE de prévision",
               title="Non-linéarité au point de fonctionnement")
    axs[2].plot([0, 1], [0, 1], "k--", lw=1)
    axs[2].set(xlabel="NRMSE réelle", ylabel="NRMSE prédite (système retiré)", title=f"Prédicteur : {model}")
    for ax in axs:
        ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig("exploration.png", dpi=110)
    milestone("E3", True, "figure enregistrée : exploration.png")
    print("=" * 84)
    if show:
        plt.show()
    return {"theta": theta, "model": model, "F": F, "perf": perf, "family": fam}


if __name__ == "__main__":
    print("Étape 16 chargée : run_catalog(EXPLORATION, 'exploration.jsonl') puis explore_analysis(rows)")


## Step 17 — `confirm2.py`

confirmation on 90 unseen systems

In [ ]:
"""
Étape 17 — Confirmation sur des systèmes JAMAIS VUS, dont deux familles nouvelles.

Tout ce qui suit a été écrit AVANT d'exécuter le moindre système de confirmation :
le catalogue, les valeurs gelées à l'étape 16 (seuil theta, modèle), les hypothèses
P2 à P5 et leurs critères chiffrés. Rien n'est ajusté après coup ; si un critère
échoue, il est rapporté comme échoué.

  P2  Un système admis mais presque linéaire (nl < 0.01) ne prévoit pas mieux qu'un
      modèle linéaire. Critère : au moins 80 % de ces systèmes ont une NRMSE >= 0.35,
      et aucun n'est sous 0.25.
  P3  Parmi les réseaux de ressorts, c'est la capacité mémoire linéarisée mc_lin qui
      ordonne les géométries (exploration : corrélation de rang -0.73 avec l'erreur,
      contre +0.13 pour d2 ; l'hypothèse initiale portait sur d2, elle a été remplacée
      ici, avant toute confirmation). Critères, sur les 40 nouvelles géométries :
      (a) corrélation de rang entre mc_lin et l'erreur <= -0.4 ;
      (b) [gardé tel que prévu, bien que l'exploration ne le favorise pas : séparation
      0.21 contre 0.28 pour le filtre binaire] avec la règle graduée « aucun mode neutre et d2 < theta », médiane des admis
      < médiane des rejetés, et au moins 75 % du meilleur quart est admis.
  P4  Le profil prédit l'erreur mieux que la simple moyenne de la famille. Critères :
      (a) erreur quadratique du modèle gelé < celle de la référence, sur l'ensemble ;
      (b) parmi les 40 ressorts, R² du modèle > 0.1.
      Référence : moyenne de la famille dans l'exploration ; pour une famille nouvelle,
      moyenne de toute l'exploration.
  P5  Le filtre binaire de l'étape 12 tient sur l'ensemble de confirmation. Critères :
      (a) médiane des admis inférieure d'au moins 0.10 à celle des rejetés ;
      (b) au moins 75 % du meilleur quart est admis.

Notebook : exécuter d'abord les cellules des étapes 1 à 16.
Puis : conf = run_catalog(CONFIRMATION, "confirmation.jsonl")     (~1 h)
       confirm2_analysis(load_rows("exploration.jsonl"), conf)
"""

import time

import numpy as np

# Notebook : pas d'import entre fichiers.

_T0 = time.perf_counter()

# ---- Valeurs gelées à l'issue de l'étape 16 (ne pas modifier) ----------------
FROZEN_THETA = 0.01          # choisis à l'étape 16, sur les 64 systèmes d'exploration
FROZEN_MODEL = "5 plus proches voisins"
NL_LINEAR = 0.01             # P2 : « presque linéaire »
# ------------------------------------------------------------------------------


def milestone(tag, ok, msg):
    status = "OK  " if ok else "FAIL"
    print(f"[{tag}] [{status}] {msg:<62} ({time.perf_counter() - _T0:5.2f} s)")
    return ok


def result(msg):
    print(f"[RES] {msg}")


CONFIRMATION = (
    [("Ressorts", g, {"amp": 0.3, "damping": 1.0}) for g in range(36, 76)]
    + [("ESN", g, {"input_scale": a}) for g in (6, 7, 8) for a in (0.01, 0.1, 1.0, 3.0)]
    + [("Chaîne", g, {"kind": _CH["a"], "amp": a}) for g in (2, 3, 4) for a in (0.03, 0.3, 1.0, 3.0, 10.0)]
    + [("Chaîne", g, {"kind": _CH[k], "amp": 0.3}) for k in "bc" for g in (2, 3)]
    + [("Retard", g, {"gamma": a}) for g in (0, 1, 2) for a in (0.05, 0.5, 2.0)]
    + [("Retard", g, {"eta": 1.5, "phi": 0.0}) for g in (0, 1)]
    + [("Coulomb répulsif", g, {"damping": d, "E0": 0.25}) for g in range(4) for d in (1.0, 2.0)]
)


def _verdict(ok):
    return "RÉUSSI" if ok else "ÉCHOUÉ"


def _quartile_admitted(adm, perf):
    best = np.argsort(perf)[:max(1, len(perf) // 4)]
    return int(adm[best].sum()), len(best)


def confirm2_analysis(explo, conf, show=True):
    print("=" * 84)
    print("Étape 17 — confirmation sur des systèmes jamais vus")
    print("=" * 84)
    assert FROZEN_THETA is not None and FROZEN_MODEL in MODELS, "geler theta et le modèle d'abord"
    perf = np.array([r["perf"] for r in conf]); fam = np.array([r["family"] for r in conf])
    binary = np.array([r["admis"] for r in conf]); nl = np.array([r["nl"] for r in conf])
    d2 = np.array([min(r["d2"], 1e3) for r in conf])
    graded = graded_admit(conf, FROZEN_THETA)
    keys_e = {r["key"] for r in explo}
    milestone("V1", len(conf) == len(CONFIRMATION) and not any(r["key"] in keys_e for r in conf),
              f"{len(conf)} systèmes, aucun en commun avec l'exploration")
    milestone("V2", bool(np.all(np.isfinite(perf))), "prévisions complètes")

    result("par famille : effectif, admis (binaire), admis (gradué), NRMSE médiane")
    for f in sorted(set(fam)):
        m = fam == f
        print(f"      {f:<17} n = {m.sum():>2}   binaire {binary[m].sum():>2}   gradué {graded[m].sum():>2}   "
              f"médiane {np.median(perf[m]):.2f}")
    out = {}

    # ---- P2
    m = binary & (nl < NL_LINEAR)
    if m.sum():
        frac, low = float(np.mean(perf[m] >= 0.35)), int(np.sum(perf[m] < 0.25))
        out["P2"] = frac >= 0.8 and low == 0
        result(f"P2 : {m.sum()} systèmes admis presque linéaires ; {frac:.0%} ont une NRMSE >= 0.35, "
               f"{low} sous 0.25 (min {perf[m].min():.2f}) -> {_verdict(out['P2'])}")
        m2 = binary & (nl >= NL_LINEAR)
        result(f"     pour comparaison, admis non linéaires : médiane {np.median(perf[m2]):.2f} (n = {m2.sum()})")
    else:
        out["P2"] = None
        result("P2 : aucun système concerné -> NON TESTÉ")

    # ---- P3
    s = fam == "Ressorts"
    mc = np.array([r["mc_lin"] for r in conf])
    rho = spearman(mc[s], perf[s])
    out["P3a"] = rho <= -0.4
    result(f"P3a : ressorts, corrélation de rang entre mc_lin et l'erreur = {rho:+.2f} (critère <= -0.40) "
           f"-> {_verdict(out['P3a'])}")
    result(f"     pour comparaison, corrélation de rang entre d2 et l'erreur : {spearman(d2[s], perf[s]):+.2f}")
    ga = graded[s]
    if ga.any() and (~ga).any():
        ma, mr = np.median(perf[s][ga]), np.median(perf[s][~ga])
        k, n = _quartile_admitted(ga, perf[s])
        out["P3b"] = ma < mr and k / n >= 0.75
        result(f"P3b : règle graduée, {ga.sum()} admis / {(~ga).sum()} rejetés ; médianes {ma:.2f} / {mr:.2f} ; "
               f"meilleur quart admis {k}/{n} -> {_verdict(out['P3b'])}")
    else:
        out["P3b"] = False
        result(f"P3b : la règle graduée ne sépare rien ({ga.sum()} admis sur {s.sum()}) -> ÉCHOUÉ")
    ba = binary[s]
    if ba.any() and (~ba).any():
        k, n = _quartile_admitted(ba, perf[s])
        result(f"     pour comparaison, filtre binaire sur les ressorts : {ba.sum()} admis ; médianes "
               f"{np.median(perf[s][ba]):.2f} / {np.median(perf[s][~ba]):.2f} ; meilleur quart admis {k}/{n}")

    # ---- P4
    Fe, ye = feature_matrix(explo), np.array([r["perf"] for r in explo])
    fe = np.array([r["family"] for r in explo])
    pred = MODELS[FROZEN_MODEL](Fe, ye, feature_matrix(conf))
    base = np.array([ye[fe == f].mean() if (fe == f).any() else ye.mean() for f in fam])
    mse_m, mse_b = float(np.mean((perf - pred) ** 2)), float(np.mean((perf - base) ** 2))
    out["P4a"] = mse_m < mse_b
    result(f"P4a : erreur quadratique, modèle gelé ({FROZEN_MODEL}) {mse_m:.4f} contre référence {mse_b:.4f} "
           f"-> {_verdict(out['P4a'])}")
    r2s = r2_score(perf[s], pred[s])
    out["P4b"] = r2s > 0.1
    result(f"P4b : parmi les ressorts, R² du modèle = {r2s:+.2f} (critère > 0.10) -> {_verdict(out['P4b'])}")
    result(f"     R² sur l'ensemble : modèle {r2_score(perf, pred):+.2f}, référence {r2_score(perf, base):+.2f} ; "
           f"corrélation de rang prédit/réel {spearman(pred, perf):+.2f}")
    for f in sorted(set(fam)):
        m = fam == f
        print(f"      {f:<17} erreur quadratique : modèle {np.mean((perf[m] - pred[m]) ** 2):.4f}   "
              f"référence {np.mean((perf[m] - base[m]) ** 2):.4f}")

    # ---- P5
    ma, mr = np.median(perf[binary]), np.median(perf[~binary])
    k, n = _quartile_admitted(binary, perf)
    out["P5a"], out["P5b"] = (mr - ma) >= 0.10, k / n >= 0.75
    result(f"P5a : filtre binaire, {binary.sum()} admis / {(~binary).sum()} rejetés ; médianes {ma:.2f} / {mr:.2f} "
           f"-> {_verdict(out['P5a'])}")
    result(f"P5b : meilleur quart admis {k}/{n} -> {_verdict(out['P5b'])}")
    for f in ("Retard", "Coulomb répulsif"):
        m = fam == f
        a, r = m & binary, m & ~binary
        print(f"      famille nouvelle {f:<17} admis {a.sum()} (médiane {np.median(perf[a]) if a.any() else np.nan:.2f})"
              f"   rejetés {r.sum()} (médiane {np.median(perf[r]) if r.any() else np.nan:.2f})")

    import matplotlib.pyplot as plt
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.3))
    marks = {"ESN": "o", "Ressorts": "^", "Chaîne": "s", "Retard": "D", "Coulomb répulsif": "X"}
    for f, mk in marks.items():
        m = fam == f
        axs[0].scatter(perf[m & binary], pred[m & binary], marker=mk, s=34, label=f)
        axs[0].scatter(perf[m & ~binary], pred[m & ~binary], marker=mk, s=34, facecolors="none", edgecolors="0.4")
        axs[1].scatter(nl[m & binary] + 1e-4, perf[m & binary], marker=mk, s=34, label=f)
    axs[0].plot([0, 1], [0, 1], "k--", lw=1)
    axs[0].set(xlabel="NRMSE réelle", ylabel="NRMSE prédite par le modèle gelé",
               title="P4 : prédiction (creux : rejetés par le filtre)")
    axs[1].axvline(NL_LINEAR, color="tab:red", lw=1)
    axs[1].set(xscale="log", xlabel="non-linéarité statique nl", ylabel="NRMSE de prévision",
               title="P2 : systèmes admis")
    axs[2].scatter(mc[s], perf[s], marker="^", s=34, c=np.where(binary[s], "tab:blue", "0.6"))
    axs[2].set(xlabel="capacité mémoire linéarisée mc_lin", ylabel="NRMSE de prévision",
               title="P3 : 40 nouveaux réseaux de ressorts (gris : rejetés binaire)")
    for ax in axs[:2]:
        ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig("confirmation2.png", dpi=110)
    milestone("V3", True, "figure enregistrée : confirmation2.png")
    n_t = sum(v is not None for v in out.values()); n_ok = sum(bool(v) for v in out.values() if v is not None)
    result(f"bilan : {n_ok} critères réussis sur {n_t} testés")
    print("=" * 84)
    if show:
        plt.show()
    return out


if __name__ == "__main__":
    print("Étape 17 chargée : run_catalog(CONFIRMATION, 'confirmation.jsonl') puis confirm2_analysis(explo, conf)")


## Step 2b — `live_time.py`

live display (needs %matplotlib widget)

In [ ]:
"""
Étape 2b — Le réservoir en direct, dans le temps.

L'entrée change de classe toutes les SEG étapes (bruit -> sinus -> logistique,
les trois signaux de M1). Le vecteur de complexité est recalculé sur une
fenêtre glissante des W derniers pas.

Notebook : exécuter d'abord les cellules des étapes 1, 2a, 3 et 4. Puis, dans une
cellule séparée, choisir un backend interactif AVANT de lancer :

    %matplotlib widget        # dans VS Code (pip install ipympl), ou :
    %matplotlib qt            # fenêtre séparée (pip install pyqt5)

    anim = live()                              # animation (espace = pause)
    anim = live(rho=1.3, alpha=0.5)            # autre régime du réservoir
    live(headless=True, frames=200)            # sans animation : texte + image finale

Garder `anim = ...` : sans variable, Python efface l'animation et elle se fige.
"""

import time

import numpy as np

# Notebook : pas d'import entre fichiers. complexity_vector, conditional_lyapunov,
# entropy_complexity (étape 1), ESNSubstrate (étape 3), normalize_vector, AXES et
# draw_axes (étape 4) viennent des cellules précédentes.

N, W, K = 50, 2000, 50       # nœuds, fenêtre glissante (pas), pas simulés par image
SEG = 3000                   # durée de chaque classe d'entrée (pas)
EPISODE, DELTA = 400, 1e-6   # lambda : durée d'un épisode, écart initial
D = 4                        # motifs ordinaux (24 motifs : assez de fenêtres en direct)
TRACE, TRAIL, PRINT_EVERY = 300, 40, 10
CLASSES = ["bruit", "sinus", "logistique"]

_T0 = time.perf_counter()


def milestone(tag, ok, msg):
    status = "OK  " if ok else "FAIL"
    print(f"[{tag}] [{status}] {msg:<62} ({time.perf_counter() - _T0:5.2f} s)")


class InputSource:
    """Entrée scalaire dans [-1, 1] qui change de classe toutes les SEG étapes."""

    def __init__(self, seed=0):
        self.rng, self.t, self.x = np.random.default_rng(seed), 0, 0.4

    def next(self):
        cls = CLASSES[(self.t // SEG) % len(CLASSES)]
        if cls == "bruit":
            u = self.rng.uniform(-1, 1)
        elif cls == "sinus":
            u = np.sin(2 * np.pi * self.t / 50)
        else:
            self.x = 4 * self.x * (1 - self.x)
            if not 0 < self.x < 1:                    # garde-fou : point fixe 0
                self.x = self.rng.uniform(0.1, 0.9)
            u = 2 * self.x - 1
        self.t += 1
        return u, cls


class LiveSystem:
    """Un substrat et sa copie décalée, avancés ensemble avec la même entrée.
    N'utilise que l'interface : marche avec n'importe quel substrat de substrates.py."""

    def __init__(self, sub, seed=0):
        self.sub, self.src = sub, InputSource(seed)
        self.rng = np.random.default_rng(seed + 1)
        self.sa = sub.init_state(self.rng)
        self._new_episode()
        self.X, self.U = np.empty((0, sub.n_obs)), np.empty(0)
        self.cls, self.lams, self.last_ep_d = CLASSES[0], [], None

    def _new_episode(self):
        self.sb = self.sub.perturb(self.sa, DELTA, self.rng)
        self.ep_a, self.ep_b = [], []

    def advance(self, k):
        xs, us = [], []
        for _ in range(k):
            u, self.cls = self.src.next()
            self.sa, self.sb = self.sub.step(self.sa, u), self.sub.step(self.sb, u)
            xa, xb = self.sub.observe(self.sa), self.sub.observe(self.sb)
            xs.append(xa); us.append(u)
            self.ep_a.append(xa); self.ep_b.append(xb)
            if len(self.ep_a) == EPISODE:             # fin d'épisode : on mesure lambda
                Xa, Xb = np.array(self.ep_a), np.array(self.ep_b)
                self.lams.append(conditional_lyapunov(Xa, Xb))
                self.last_ep_d = np.linalg.norm(Xa - Xb, axis=1)
                self._new_episode()
        self.X = np.vstack([self.X, xs])[-W:]
        self.U = np.concatenate([self.U, us])[-W:]

    def measure(self):
        """Vecteur sur la fenêtre + position (H, C) de l'entrée. None tant que la fenêtre n'est pas pleine."""
        if len(self.X) < W:
            return None
        return complexity_vector(self.X, D=D, warn=False), entropy_complexity(self.U, D, warn=False)


def live(rho=0.9, alpha=0.3, scale=1.0, headless=False, frames=200):
    """Lance le direct. Renvoie l'animation (à garder dans une variable)."""
    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation

    sub = ESNSubstrate(N=N, rho=rho, alpha=alpha, input_scale=scale)
    sysm = LiveSystem(sub)
    milestone("L1", True, f"réservoir créé : N={N}, rho={rho}, alpha={alpha}, entrée x{scale}")

    # Repères du plan de Rosso : les trois signaux de M1
    rng, T = np.random.default_rng(1), 20000
    logi = np.empty(T); logi[0] = 0.4
    for t in range(T - 1):
        logi[t + 1] = 4 * logi[t] * (1 - logi[t])
    refs = {"bruit": rng.standard_normal(T), "sinus": np.sin(2 * np.pi * np.arange(T) / 50),
            "logistique": logi}
    refs = {k: entropy_complexity(v, D, warn=False) for k, v in refs.items()}
    milestone("L2", True, "repères : " + ", ".join(f"{k} ({h:.2f}, {c:.2f})" for k, (h, c) in refs.items()))

    # ------------------------------------------------------------- la figure
    fig = plt.figure(figsize=(13, 11))
    gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1.1])
    axA, axB = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])
    axC, axD = fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])
    axP = fig.add_subplot(gs[2, :])
    lnU, = axA.plot([], [], "k", lw=1.5, label="entrée u")
    lnX = [axA.plot([], [], lw=0.8, label=f"nœud {i}")[0] for i in range(4)]
    axA.set(xlim=(0, TRACE), ylim=(-1.1, 1.1), xlabel="derniers pas", title="Traces")
    axA.legend(loc="upper right", fontsize=7)

    for k, (h, c) in refs.items():
        axB.plot(h, c, "s", color="grey"); axB.annotate(k, (h, c), fontsize=8, xytext=(4, 4),
                                                        textcoords="offset points")
    trail, = axB.plot([], [], "-", color="tab:blue", alpha=0.4)
    ptR, = axB.plot([], [], "o", color="tab:blue", ms=9, label="réservoir")
    ptU, = axB.plot([], [], "x", color="k", ms=9, mew=2, label="entrée")
    axB.set(xlim=(0, 1.02), ylim=(0, 0.5), xlabel="H (entropie)", ylabel="C_JS (complexité)",
            title="Plan complexité–entropie")
    axB.legend(loc="upper left", fontsize=8)

    lnD, = axC.plot([], [], "tab:red")
    axC.set(xlim=(0, EPISODE), ylim=(-16, 1), xlabel="pas dans l'épisode",
            ylabel="log10 écart", title="Divergence des deux copies (dernier épisode)")
    txtC = axC.text(0.02, 0.05, "", transform=axC.transAxes, fontsize=9)

    hist = {"H_perm": [], "C_JS": [], "PR": [], "C_N_form": []}
    lnH = {k: axD.plot([], [], label=k)[0] for k in hist}
    steps = []
    axD.set(ylim=(0, 1.05), xlabel="pas", title="Évolution du vecteur")
    axD.legend(loc="upper left", fontsize=8)
    txtD = axD.text(0.98, 0.95, "", transform=axD.transAxes, ha="right", va="top",
                    fontsize=8, family="monospace")

    # Coordonnées parallèles (étape 4) : ligne actuelle épaisse, passé estompé
    PAST = 30
    COLORS = {"bruit": "tab:blue", "sinus": "tab:orange", "logistique": "tab:green"}
    draw_axes(axP)
    i_lam = [k for k, _, _ in AXES].index("lam")
    axP.text(i_lam + 0.15, 0.52, "lambda = 0", fontsize=7, color="tab:red")
    past_lines = [axP.plot([], [], lw=1)[0] for _ in range(PAST)]
    cur_line, = axP.plot([], [], "o-", lw=3, ms=7, color="k")
    cur_txt = [axP.text(i + 0.06, 0, "", fontsize=8, va="center") for i in range(len(AXES))]
    axP.set_title("Coordonnées parallèles — couleur = entrée (bleu : bruit, orange : sinus, "
                  "vert : logistique), passé estompé")
    past = []                                   # (vecteur normalisé, classe d'entrée)

    state = {"paused": False, "frame": 0, "full": False, "lam": False, "cls": CLASSES[0],
             "t_last": time.perf_counter()}
    fig.canvas.mpl_connect("key_press_event",
                           lambda e: state.update(paused=not state["paused"]) if e.key == " " else None)

    def update(_):
        if state["paused"]:
            return
        sysm.advance(K)
        state["frame"] += 1
        t = sysm.src.t

        n = min(TRACE, len(sysm.U))
        lnU.set_data(np.arange(n), sysm.U[-n:])
        for i, ln in enumerate(lnX):
            ln.set_data(np.arange(n), sysm.X[-n:, i])

        if sysm.last_ep_d is not None:
            lnD.set_data(np.arange(EPISODE), np.log10(np.maximum(sysm.last_ep_d, 1e-16)))
            lam = sysm.lams[-1]
            txtC.set_text(f"lambda = {lam:+.4f} / pas   ({'écho' if lam < 0 else 'chaos'})")
            if not state["lam"]:
                state["lam"] = True
                milestone("L4", np.isfinite(lam), f"premier lambda mesuré : {lam:+.4f}")

        if sysm.cls != state["cls"]:
            print(f"   --> t={t}: l'entrée passe de « {state['cls']} » à « {sysm.cls} »")
            state["cls"] = sysm.cls

        res = sysm.measure()
        if res is None:
            return
        C, (Hu, Cu) = res
        if not state["full"]:
            state["full"] = True
            milestone("L3", True, f"fenêtre pleine ({W} pas) : premières mesures")
        if not all(np.isfinite(v) for k, v in C.items() if k not in ("lam", "consist")):
            milestone("L5", False, f"t={t} : NaN dans le vecteur {C}")

        steps.append(t)
        for k in hist:
            hist[k].append(C[k]); lnH[k].set_data(steps, hist[k])
        axD.set_xlim(max(0, t - 30 * SEG), t + 1)
        trail.set_data(hist["H_perm"][-TRAIL:], hist["C_JS"][-TRAIL:])
        ptR.set_data([C["H_perm"]], [C["C_JS"]]); ptU.set_data([Hu], [Cu])
        lam = sysm.lams[-1] if sysm.lams else np.nan
        tau = f"{C['tau_med']:8.1f}" if C["tau_med"] < 500 else "  >= 500"     # 500 = max_lag : plafonné
        txtD.set_text(f"C_N      {C['C_N']:8.2f}\ntau_med  {tau}\n"
                      f"tau_sprd {C['tau_sprd']:8.2f}\nlambda   {lam:+8.4f}")
        fig.suptitle(f"t = {t}   entrée : {sysm.cls}   (espace = pause)")

        v = normalize_vector(C | {"lam": lam})
        past.append((v, sysm.cls)); del past[:-PAST]
        for j, ln in enumerate(past_lines):
            if j < len(past) - 1:                   # la dernière est la ligne actuelle
                pv, pc = past[j]
                ln.set_data(range(len(AXES)), pv)
                ln.set_color(COLORS[pc]); ln.set_alpha(0.05 + 0.35 * j / PAST)
            else:
                ln.set_data([], [])
        cur_line.set_data(range(len(AXES)), v); cur_line.set_color(COLORS[sysm.cls])
        raw = [f"{C['H_perm']:.2f}", f"{C['C_JS']:.3f}", f"{C['PR']:.3f}", f"{C['C_N_form']:.2f}",
               tau.strip(), f"{C['tau_sprd']:.2f}", f"{lam:+.3f}",
               "non mesurée"]                       # le direct ne lance pas de 3e trajectoire
        for i, (txt, r) in enumerate(zip(cur_txt, raw)):
            txt.set_position((i + 0.06, v[i] if np.isfinite(v[i]) else 0.5)); txt.set_text(r)

        if state["frame"] % PRINT_EVERY == 0:
            now = time.perf_counter()
            fps, state["t_last"] = PRINT_EVERY / (now - state["t_last"]), now
            print(f"[t={t:6d}] {sysm.cls:<10} H={C['H_perm']:.3f} C={C['C_JS']:.3f} "
                  f"PR={C['PR']:.3f} C_N={C['C_N']:6.1f} forme={C['C_N_form']:.2f} tau={tau.strip():>6} "
                  f"lam={lam:+.4f}  ({fps:4.1f} img/s)")

    fig.tight_layout(rect=(0, 0, 1, 0.96))    # place pour le titre général
    if headless:
        for i in range(frames):
            update(i)
        fig.savefig("live_snapshot.png", dpi=110)
        plt.close(fig)                         # évite un double affichage dans le notebook
        milestone("L6", True, f"{frames} images simulées, capture : live_snapshot.png")
        return None
    anim = FuncAnimation(fig, update, interval=1, cache_frame_data=False)
    plt.show()
    return anim


## Re-analyse the saved runs without re-simulating

The exploration and confirmation runs (about 25 min each) are saved in `data/`.

In [ ]:
explo = load_rows("../data/exploration.jsonl")
frozen = explore_analysis(explo)
conf = load_rows("../data/confirmation.jsonl")
criteria = confirm2_analysis(explo, conf)
